# Visualizing and Understanding Neural Models in NLP  

## Abstract  

虽然神经网络已经成功应用于许多 NLP 任务，但由此产生的基于向量的模型很难解释。例如，他们如何实现组合性并不清楚，从单词和短语的含义来构建句子意义。在本文中，我们描述了用于 NLP 的神经模型中的组合性可视化的策略，其灵感来源于类似的计算机视觉工作。我们首先绘制 unit values ，用以表示negation,，强化和 concessive clauses 的组合性，从而使我们能够在 negation, 中看到众所周知的标记不对称。然后，我们介绍一些单元显著性的可视化方法，即从一阶导数对最终构成含义的贡献量。我们的通用方法在理解深度网络的组合性和其他语义属性方面可能有广泛的应用。  

## 1 Introduction  

神经模型在各种 NLP 任务中匹配或超越其他最先进系统的性能。然而，与传统的基于特征的分类器不同，这些分类器对各种可解释特征（词类，命名实体，单词形状，句法分析特征等）进行赋值和优化，深度学习模型的行为不太容易解释。深度学习模型主要通过多层神经结构对词嵌入（低维，连续，实值向量）进行操作，其每一层都被表征为一组隐藏的神经元单元。目前还不清楚深度学习模型是如何处理 composition,，实现 negation or intensification 等功能，或者结合句子不同部分的意义，建立句子意义。  

在本文中，我们探索了多种策略来解释神经模型中的意义构成。我们采用传统方法如表示性绘图，并引入简单的策略来测量神经单元对意义构成的贡献程度，它的“显著性”或使用一阶导数的重要性。  

本工作中呈现的可视化技术/模型揭示了神经模型的工作原理：例如，我们说明 LSTM 的成功是因为它能够比其他模型更加关注重要的关键词；多个composition 中的构成具有竞争性，并且这些模型能够捕捉负面的不对称，这是在自然语言理解中构成语义的重要属性；存在尖锐的空间局部性，某些维度以令人惊讶的定位方式标记否定和量化。尽管我们的尝试只涉及神经模型中的 superficial points 问题，并且每种方法都有其优缺点，但它们可以提供一些洞察神经模型在基于语言的任务中的行为，标志着在自然语言处理中理解它们如何实现有意义的 composition。  

下一节将介绍一些视觉和自然语言处理中的可视化模型，这些启发了这项工作。我们在第 3 节描述数据集和采用的神经模型。不同的可视化策略和相应的分析结果分别在第 4,5,6 节中给出，接下来是一个简短的结论。  

## 2 A Brief Review of Neural Visualization  

![Aaron Swartz](https://github.com/liyibo/cv_notebooks/blob/master/markdown_pics/NLP/1/2.jpg?raw=true)

通常通过将嵌入空间投影到两个维度并观察到类似的词倾向于聚集在一起，相似性通常通过图形可视化。（Karpathy et al。，2015）试图从统计观点来解释 recurrent 神经模型。其他相关尝试包括（Fyshe 等，2015；Faruqui 等，2015）。  

用于解释和可视化神经模型的方法在视觉中得到了更为重要的探索，尤其是对于卷积神经网络（CNN 或 ConvNets）（Krizhevsky等，2012），其中图像像素的原始矩阵被多层神经网络卷积并汇集到隐藏层。这些方法包括：  

（1）反演：通过训练一个额外的模型来反演表征，从不同的神经层次反馈到初始输入图像。重建背后的直觉是，可以从当前表示重构的像素内容。反演算法允许当前表示与原始图像的相应部分对齐。  

（2）反向传播和 Deconvolutional Networks：错误从输出层反向传播到每个中间层，最后传播到原始图像输入。反卷积网络以类似的方式工作，将输出逐层投影回初始输入，每一层与一个监督模型相关联，用于将较高的映射投影到较低的映射。这些策略使得有可能发现活跃区域或对最终贡献最大的区域。  

（3）生成：这组作品根据已经训练过的神经模型指导的草图生成特定类别的图像。模型以随机初始化的图像开始。在图像构建的不同阶段激活的特定图层可以帮助解释。  

尽管上述策略激发了我们在本文中提出的工作，但与 NLP 之间存在根本性差异。在 NLP 中词作为基本单位，因此（单词）向量而不是单个像素是基本单位。词的序列（例如，短语和句子）也以比像素的排列更结构的方式呈现。与我们的研究并行，Karpathy 等通过分析循环神经模型的预测和误差，从错误分析的角度探索了类似的方向。  

## 3 Datasets and Neural Models  

我们研究了两个神经模型训练的数据集，其中一个数据集规模相对较小，另一个规模较大。  

### 3.1 Stanford Sentiment Treebank  

斯坦福大学 Sentiment Treebank 是广泛用于神经模型评估的基准数据集。数据集包含每个分析树组件的 gold-standard  情绪标签，从句子到短语到单个词，用于 11855 个句子中的 215154 个短语。任务是在短语和句子两个层次上执行细粒度（非常积极，积极，中性，消极和非常消极）和粗粒度（积极与消极）分类。有关数据集的更多详细信息，请参阅 Socher et al。  

虽然这个数据集的许多研究都使用递归分析树模型，但在这项工作中，我们只使用标准序列模型（RNN 和 LSTM），因为这些模型是当前最广泛使用的神经模型，并且顺序可视化更直接。因此，我们首先将每个分析树节点转换为一个 tokens 序列。该序列首先映射到短语/句子表示并馈入 softmax 分类器。短语/句子表示使用以下三种模式构建：具有 TANH 激活函数的标准 Recurrent 序列，LSTM 和双向 LSTM。 有关这三种模型的详细信息，请参阅附录。  

### 3.2 Sequence-to-Sequence Models  

seq2seq 是旨在生成给定输入的输出文本序列的神经模型。从理论上讲，seq2seq 模型可以适应 NLP 任务，这些任务可以为给定输入的预测输出，并且由于不同的输入和输出而用于不同的目的，例如机器翻译，其输入对应于源语句和输出到目标语句；如果输入对应于消息并且输出对应于响应，则可用于对话。seq2seq 需要接受大量数据的训练，以获得学习对之间的隐式语义和句法关系。  

seq2seq 模型使用 LSTM 模型将输入序列映射到向量表示，然后基于预先获得的表示顺序预测 tokens。该模型定义了输出（Y）上的分布，并使用 softmax 函数依次预测给定输入（X）的 tokens。  

## 4 Representation Plotting  

我们从简单的 plots 开始，使用斯坦福情绪树库（Stanford Sentiment Treebank）揭示 local compositions。  

![Aaron Swartz](https://github.com/liyibo/cv_notebooks/blob/master/markdown_pics/NLP/1/1.jpg?raw=true)

**Local Composition** 图1 显示了一个 60d 热图矢量，用于表示选定的单词/短语/句子，重点在于 extent 修改（状语和 adjectival）和 negation。短语或句子的嵌入是通过组合来自预训练模型的词表达来实现的。  

我们在图2 中使用 tsne（Van der Maaten和Hinton，2008）对单词和短语进行可视化，故意添加一些随机单词以供比较。可以看出，神经模型很好地学习了 local compositionally 性质，将否定词+积极词（“不好”，“不好”）与负面词组合在一起。还要注意否定的不对称性：“not bad”更多的和负性词在聚在一起（如图1和图2所示）。这种不对称性在语言学中得到了广泛的讨论，例如由于显著性而引起的，因为'好'是该尺度的未标记方向。这表明，虽然模型在图1 中似乎集中于某些 negation，但神经模型不仅仅是学习将固定变换应用于“不”，而是能够捕捉不同单词构成中的细微差异。  

**Concessive Sentences ** 在让步性句子中，两个从句具有相反的词性，通常与期望相反的含义相关。图3 绘制出两个让步从句随时间演变的表示图。这些图表明：  

1.对于目标是预测特定语义维度（与语言模型词预测等一般任务相反）的情感分析等任务而言，过大的维度会导致许多维度无法运行（值接近0），从而导致两个维度 相反情绪的句子仅在几个维度上有所不同。 这可以解释为什么更多的维度不一定会导致这些任务更好的表现（例如，如（Socher等人，2013）中所报道的，当词维度设置在25和35之间时达到最佳性能）。  

。。。  

**Clause Composition** 在图4 中，我们更详细地探讨了此 clause composition。通过添加负面 clauses，比如“although it had bad acting”，或者““but it is too long”到简单的积极的“I like the movie”的结尾，这些表现力更接近负面情绪区域。相比之下，为负面 clause 增加一个concessive clause 并不会朝着积极的方向发展；“I hate X but ...”仍然是非常消极的，与“I hate X”没有什么不同。这种差异再次表明该模型能够捕捉负面不对称。  

## 5 First-Derivative Saliency  

![Aaron Swartz](https://github.com/liyibo/cv_notebooks/blob/master/markdown_pics/NLP/1/3.jpg?raw=true)

在本节中，我们描述了另一种策略，它受到视觉中的后向传播策略的启发。它衡量每个输入单位对最终决策的贡献量，可以用一阶导数来近似。  

更正式地说，对于分类模型，输入 E 与标准类别标签 c 相关联。（根据NLP任务，输入可以是单词或单词序列的嵌入，而标签可以是 POS 标签，情感标签，预测的下一个单词索引等）给定输入词的嵌入 E 以及关联标签 c，训练后的模型将该对（E，c）与 $S-c(E)$ 分数联系起来。目标是决定 E 的哪些units 对 $S-c(E)$ 做出最重要的贡献，并因此决定是否选择类别标签 c。   

### 5.1 Results on Stanford Sentiment Treebank  

![Aaron Swartz](https://github.com/liyibo/cv_notebooks/blob/master/markdown_pics/NLP/1/4.jpg?raw=true)

我们首先在斯坦福树库上说明结果。我们在图5，图6 和图7 中绘制了三个句子的显著性分数，将训练的模型应用于每个句子。每行对应于表示每个维度的每个网格的对应词表示的显著性分数。这些例子是基于明确的情绪指标“hate”，这些指标给了他们所有的负面情绪。  

**“I hate the movie”** 所有这三种模式都高度赞同“hate”，并削弱了其他 tokens 的影响力。LSTM 比标准的循环模型更加关注“hate”，但双向LSTM 显示最清晰的焦点，几乎不重视“hate”以外的单词。这大概是由于 LSTM 和 Bi-LSTM 中的门结构控制信息流，使得这些体系结构更好地滤除不太相关的信息。  

**“I hate the movie that I saw last night”** 所有三种模式都指定了正确的观点。简单的循环模型在筛选出不相关的信息方面再次做得很差，对与情绪无关的单词分配过多的显著性。然而，尽管这个句子更长，但没有一个模型受到梯度消失问题的影响；“hate”的显著性在卷积操作之后仍然突出。  

**““I hate the movie though the plot is interesting”** 简单的循环模型只强调第二个 clause “the plot is interesting”，并没有对第一个 clause “I hate the movie”给予评价。这看起来可能是由渐变消失引起的，但该模型正确地将该句子分类为非常消极，表明它成功合并来自第一个否定句子的信息。我们单独测试了单个 clause “though the plot is interesting”。标准循环模型自信地将其标记为正面。因此，尽管第一个句子中的单词显著性得分较低，但简单的循环系统设法依赖该句子，并且从后面的肯定句子中淡化信息 - 尽管后面的单词具有较高的显著性分数。这说明了显著性可视化的局限性。一阶导数不能捕获所有我们想要显示的信息，也许是因为它们只是粗略地接近 individual 贡献，并且可能不足以处理高度非线性的情况。相比之下，LSTM 强调第一个 clause，大大减弱了第二个 clause 的影响，而 Bi-LSTM 侧重于“hate the movie”和“plot is interesting”。  

### 5.2 Results on Sequence-to-Sequence Autoencoder  

![Aaron Swartz](https://github.com/liyibo/cv_notebooks/blob/master/markdown_pics/NLP/1/6.jpg?raw=true)

图9 表示在每个时间步骤预测对应 token 方面自动编码器的显著热图。随着解码的进行，我们通过反向传播计算每个前一个词的一阶导数。每个网格对应于每个 1000 维单词向量的平均显著性值的大小。热图给出了解码过程中神经模型行为的清晰概览。观察总结如下：  

1.对于每个预测单词的时间步长，seq2seq 模型设法将单词预测返回到输入端的对应区域（自动学习对齐），例如，当需要预测的 token 为“hate”时，以 token “hate”为中心的输入区域会产生更大的影响。  

2.神经解码将先前构建的表示与当前步骤中预测的单词相结合。随着解码的进行，初始输入对解码（即源句子中的tokens）的影响逐渐减小，因为更多的预先预测的单词被编码在矢量表示中。与此同时，语言模型的影响逐渐占据主导地位：当预测单词“boring”时，模型更重视早期预测 tokens "plot" 和 "is"，但对输入中对应的区域较少关注，eg. 输入中的单词“boring “。  

## 6 Average and Variance  

![Aaron Swartz](https://github.com/liyibo/cv_notebooks/blob/master/markdown_pics/NLP/1/5.jpg?raw=true)

对于将单词嵌入视为参数以从头开始优化的设置（与使用预先训练的嵌入相反），我们提出了第二个令人惊讶的简单而直接的方法来可视化重要指标。我们首先计算句子中所有单词的单词嵌入的平均值。一个词的显著性或影响力的衡量标准是它偏离这个平均值。这个想法是，在训练过程中，模型会学习使 indicators 与 non-indicator 词不同，即使在经过多层计算之后，它们也能够脱颖而出。  

图8 显示了方差图；每个网格对应于 $\parallel e_{ij} - \frac {1}{N_s} \sum_{i' \in N_S} e_{i'j} \parallel^2$ 的值，其中 $e_{ij}$ 表示单词 i 的第 j 维的值，N 表示句子内 token 的数量。  

如图所示，基于方差的显著性度量也很好地强调了相关的情感词。该模型确实有缺点：（1）它只能用于字嵌入是学习参数的场景（2）很明显，模型能够很好地显示局部组合性。  

## 7 Conclusion  

在本文中，我们提供了几种方法来帮助可视化和解释神经模型，了解神经模型如何构成 meanings，展示 negation 的不对称性并在某些方面解释 LSTM 在这些任务中的强大性能。  

尽管我们的尝试只是触及神经模型中的表面观点，并且每种方法都有其优点和缺点，但它们可以一起提供有关神经模型在基于语言的任务中的行为的一些见解，标志着理解它们如何在自然语言处理获得意义组成的初始步骤。我们未来的工作包括使用可视化结果来执行错误分析，以及了解不同神经模型的强度限制。  

# Distributed Representations of Words and Phrases and their Compositionality  

## Abstract  

最近引入的连续 Skip-gram 模型是学习高质量 distributed vector representations(分布向量表示)的有效方法，distributed vector representations可以捕获大量精确的句法和语义关系。在本文中，我们提出了几个扩展，提高了向量的质量和训练速度。通过对 frequent words 进行二次抽样，我们获得了显著的加速，同时还学习了更多的 regular word representations(常规单词表示)。我们还提出了一个分层 softmax 的简单替代方案，称为 negative sampling (负采样)。word representations 的一个固有限制是：它们不关心词序，而且无法表示 idiomatic phrases(习惯用语)。例如，不能简单地将“Canada/加拿大”和“Air/空中”的含义组合起来得到“Canada Air/加拿大航空公司”的含义。在这个例子的启发下，我们提出了一种在文本中查找短语的简单方法，并表明学习数百万个 phrases 的 good vector representations 是可能的。  

## 1 Introduction  

![Aaron Swartz](https://github.com/liyibo/cv_notebooks/blob/master/markdown_pics/NLP/2/1.jpg?raw=true)

通过分组相似的单词，在向量空间中的 distributed representations 可以帮助学习算法在 NLP 任务中获得更好的表现。最早使用 word representations 可以追溯到 1986 年(Rumelhart，Hinton 和 Williams)。这个想法已经被应用于统计语言建模且取得了相当大的成功。后续工作包括应用于自动语音识别和机器翻译，以及大范围的 NLP 任务。  

最近，Mikolov 等人引入了 Skip-gram 模型，这是一种从大量非结构化文本数据中学习高质量向量表示的有效方法。与过去大部分用于学习 word vectors 的神经网络架构不同，Skip-gram 模型的训练(参见图1)不涉及密集矩阵的乘法。这使得训练非常高效：一个优化过的单机实现可以在一天内训练超过 1000 亿字。  

使用神经网络计算的 word representation 非常有趣，因为已训练的向量明确地编码了许多语言规律和模式。有点令人惊讶的是，许多这些模式可以表示为线性变换。例如，向量计算 vec("Madrid") - vec("Spain") + vec("France") 的结果比任何其他 word vector 更接近于 vec("Paris")。   

在本文中，我们提出了原始 Skip-gram 模型的几个扩展。在训练过程中，对 frequent words 进行二次采样会导致显著的加速（大约2-10倍），并提高频率较低的 word representation 的准确性。此外，我们提出了一种用于训练 Skip-gram 模型的简化 NCE(Noise Contrastive Estimation/噪声对比估计)。结果表明，与更复杂的分层 softmax 相比，它有更快的训练速度，而且 frequent words 的 vector representation 也更好。   

words representation 天生受限于 idiomatic phrases 的表示。例如，“Boston Globe/波士顿环球报”是报纸，它不是“Boston/波士顿”和“Globe/地球”的含义的自然组合。因此，用向量来表示整个短语会使 Skip-gram 模型更具表现力。其他旨在通过组合单词向量（例如递归自动编码器/recursive autoencoders）来表示句子意义的技术也将受益于使用 phrase vectors 而不是 word vectors。   

模型从基于单词扩展到基于短语模型相对简单。首先，我们使用 data-driven 的方法识别大量的短语，然后在训练过程中将短语视为单独的 tokens (标记)。为了评估短语向量的质量，我们开发了一个包含单词和短语的类比推理任务测试集。测试集中一个典型类比对是 "Montreal":"Montreal Canadiens" :: "Toronto":"TorontoMaple Leafs" 如果最靠近 vec("Montreal Canadiens") - vec("Montreal") + vec("Toronto") 的表达是 vec("Toronto Maple  Leafs")，则被认为回答正确。   

最后，我们描述了 Skip-gram 模型的另一个有趣属性。我们发现简单的向量加法通常可以产生有意义的结果。例如，vec("Russia") + vec("river") 接近  vec("Volga River")，而 vec("Germany") + vec("capital") 接近 vec("Berlin")。这种组合性表明，通过对 word vector representation 使用基本的数学运算，可以获得不明显(non-obvious)程度的语言理解。  

## 2 The Skip-gram Model  

Skip-gram 模型的训练目标是找到可用于预测句子或文档中 surrounding words 的 word representation。更正式地，给出训练词 $w_1,w_2,,...,w_T$，Skip-gram 模型的目标是最大化对数概率：   

$\frac {1}{T} \sum_{t=1}^{T} \sum_{-c \leq j \leq c, j \neq 0} \log p(w_{t+j}|w_t) \tag{1}$

其中 c 是训练上下文（可以是中心单词 $w_t$ 的一个函数）的大小。较大的 c 意味着更多的训练样本，因此可以导致更高的准确性，同时也意味着更多的训练时间。基本 Skip-gram 公式使用 softmax 函数定义 $p(w_{t+j}|w_t)$:   

$p(w_O|w_I) = \frac{\exp (v'^T_{w_O} v_{w_I})}{\sum_{w=1}^W \exp (v'^T_{w_O} v_{w_I})} \tag{2}$

其中 $v_w$ 和 $v'_w$ 分别为 w 的输入和输出向量表示，W 为词汇表中的单词数。这个公式是不切实际的，因为计算 $\Delta \log p(w_O|w_I)$ 的花费与 W 成正比，通常会达到 $10^5 - 10^7$ 的数量级。  

### 2.1 Hierarchical Softmax  

Hierarchical(分层) softmax 是完全 softmax 的近似有效计算。它首先由 Morin 和 Bengio 在神经网络语言模型的上下文中引入。它的主要优点是，不需要评估神经网络中的 W 个输出节点以获得概率分布，仅需要评估约 log2(W) 个节点。  

分层 softmax 使用二叉树表示输出层，其中 W 个字作为其叶子节点，并且对于每个节点，显式地表示其子节点的相对概率。这些定义了一个可将概率分配给单词的 random walk(随机游走)。  

更准确地说，从一条合适的路径，可以从 root 到达每个单词 w。设 $n(w,j)$ 为从 root 到单词 w 的路径上的第 j 个节点，L(w) 为该路径的长度，则 $n(w,1) = root, n(w,L(w)) = w$。另外，对每个内节点，设 ch(n) 为 n 的 arbitrary fixed child，如果 x 是 true 则 [x]=1，否则 [x]=-1。则分层softmax 将按照如下公式定义 $p(w_O|w_I)$:   

$p(w|w_I) = \prod_{j=1}^{L(w)-1} \sigma ([n(w, j + 1) = ch(n(w, j))] \cdot v'^T_{n(w,j) v_{wI}}) \tag{3}$

其中 $\sigma(x) = 1/(1+\exp(-x))$。可以证明的是 $\sum_{w=1}^W p(w|w_I) = 1$。这意味着计算 $\log p(w_O|w_I)$ 和 $\Delta \log p(w_O|w_I)$ 的消耗与 $L(w_O)$ 成正比，通常来说不超过 $\log W$。此外，不像 Skip-gram 的标准 softmax 公式会把两个表示 $v'_w$ 和 $v_w$ 分配给每个单词 w，在分层 softmax 公式中每个单词 w 有一个 $v_w$ 且二叉树的每个内部节点 n 有一个 $v'_n$。   

分层 softmax 使用的树结构对性能有相当大的影响。Mnih 和 Hinton 探索了构建树结构的一些方法以及训练时间和结果模型精度的影响。在我们的工作中，我们使用一个霍夫曼树（binary Huffman tree），因为它将短 codes 分配给高频词，从而加快了训练速度。之前已经观察到，根据出现频率组合单词可以很好的作为基于神经网络的语言模型的一种简单加速技术。  

### 2.2 Negative Sampling  

分层 softmax 的替代方案是噪声对比估计（NCE），由 Gutmann 和 Hyvarinen 引入，并由 Mnih 和 Teh 用于语言建模。NCE 认为一个好的模型应该能够通过 logistic regression 来区分数据和噪声。这类似于 Collobert 和 Weston 使用的折页损失/hinge loss，他们通过对噪声上的数据进行排名来训练模型。  

虽然 NCE 可以最大化 softmax 的对数概率，但是 Skip-gram 模型只关注学习高质量的向量表示，因此只要向量表示保持其质量，我们可以随意简化 NCE。我们通过以下公式定义 Negative Sampling（NEG）:   

$\log \sigma(v'^T_{w_O} v_{w_I}) + \sum_{i=1}^k E_{w_i - P_n(w)} [\log \sigma(-v'^T_{w_i} v_{w_I})] \tag{4}$

用于替换 Skip-gram 中的每个 $\log p(w_O|w_I)$ 项。因此，任务是使用逻辑回归来从噪声分布 $P_n(w)$ 中区分目标单词 WO，其中每个数据样本有 k 个负样本。我们的实验表明，k 值在 5-20 范围内对于小型训练数据集是有用的，而对于大型数据集，k 可以小至 2-5。负采样和 NCE 的主要区别在于 NCE 需要样本和噪声分布的数字概率，而负采样只使用样本。尽管 NCE 使 softmax 的对数概率近似最大化，但这个属性对于我们的应用并不重要。  

NCE 和 NEG 均具有作为自由参数的噪声分布 $P_n(w)$。我们发现 unigram distribution U(w) 可以提高 NCE 和 NEG 的性能。   

### 2.3 Subsampling of Frequent Words  

在非常大的语料库中，最常见的单词可能容易出现数亿次（例如“in”，“the”和“a”）。这些单词通常比罕见单词提供更少的信息价值。例如，虽然 Skip-gram 模型受益于观察“France”和“Paris”的共现，但由于观察到“France”和“the”的频繁共现，因此几乎每一个字 - 在“the”的句子中频繁发生。这个想法也可以用于相反的方向；训练数百万个例子后，频繁词的向量表示没有显著变化。  

为了解决罕见词和常见词之间的不平衡问题，我们使用了一种简单的子采样方法：训练集中的每个词 wi 按以下公式计算的概率被丢弃  

$P(w_i) = 1 - \sqrt {\frac{t}{f(w_i)}} \tag{5}$

其中 $f(w_i)$ 是单词 wi 的频率，t 是一个选定的阈值，一般在 $10^5$ 左右。我们选择这个子采样公式是因为它在保留频率排序的同时积极地抽样频率大于 t 的词。虽然这个子采样公式是启发式选择的，但我们发现它在实践中运作良好。它会加速学习，甚至可以显著提高罕见单词的学习向量的准确性，这将在以下各节中介绍。  

## 3 Empirical Results  

![Aaron Swartz](https://github.com/liyibo/cv_notebooks/blob/master/markdown_pics/NLP/2/2.jpg?raw=true)

在本节中，我们评估的 Hierarchical Softmax（HS），噪声对比估计，Negative Sampling 和训练单词子采样。我们使用 Mikolov 等人介绍的类比推理任务。该任务由类似于“Germany” : “Berlin” :: “France” : ? 的类比组成，通过找到一个向量 x 来解决，vec（x）的余弦距离最接近 vec(“Berlin”) - vec(“Germany”) + vec(“France”)。如果 x 是“Paris”，这个特定的例子被认为已经被正确回答。该任务有两大类：句法类比（如“quick” : “quickly” :: “slow” : “slowly”）和语义类比，如从国家到首都的关系。  

![Aaron Swartz](https://github.com/liyibo/cv_notebooks/blob/master/markdown_pics/NLP/2/3.jpg?raw=true)

为了训练 Skip-gram 模型，我们使用了一个由各种新闻文章组成的大型数据集（一个 Google 内部数据集，有十亿字）。我们在训练数据中丢掉了所有出现少于 5 次的单词，结果产生了一个大小为 692K 的词汇。表1 中报告了各种 Skip-gram 模型对词类比测试集的性能。该表显示 Negative Sampling 在类比推理任务上优于 Hierarchical Softmax，并且性能甚至比噪声对比估计略好。频繁词的二次采样提高了几倍的训练速度，并使词表达更加准确。  

可以说，skip-gram 模型的线性使得它的向量更适合于这种线性类比推理，但 Mikolov 等人[8]也表明，随着训练数据量的增加，由标准 S 形回归神经网络（高度非线性）学习的向量显著改善了这一任务，这表明非线性模型对线性结构的 word representations 也有帮助。  

## 4 Learning Phrases  

![Aaron Swartz](https://github.com/liyibo/cv_notebooks/blob/master/markdown_pics/NLP/2/4.jpg?raw=true)

正如前面所讨论的，许多短语的含义并不是单个单词含义的简单组合。为了学习短语的矢量表示，我们首先找到经常出现在一起的单词，并且很少出现在其他上下文中。 例如，“纽约时报”和“多伦多枫叶队”在训练数据中被独特的 tokens 所取代，而“this is”这个 bigram 将保持不变。  

这样，我们可以形成很多合理的短语，而不会大大增加词汇的大小；从理论上讲，我们可以使用所有的 n 元语言来训练 Skip-gram 模型，但是这会占用过多的内存。之前已经开发了许多技术来识别文本中的短语；然而，我们的工作范围不在于比较它们。我们决定使用一种简单的数据驱动方法，在这种方法中，短语是基于 unigram 和 bigram 计数形成的  

$score(w_i,w_j) = \frac {count(w_i,w_j) - \delta}{count(w_i) \times count(w_j)} \tag{6}$

δ被用作 discounting 系数，并且防止形成包含非常罕见单词的太多短语。然后将具有高于所选阈值的分数用作短语。通常，我们在阈值下降的情况下对训练数据运行 2-4 次，允许形成由多个单词组成的更长的短语。我们使用涉及短语的新类比推理任务来评估短语表示的质量。表 2 显示了这个任务中使用的五类比喻的例子。该数据集可在 Web2 上公开获得。  

### 4.1 Phrase Skip-Gram Results  

![Aaron Swartz](https://github.com/liyibo/cv_notebooks/blob/master/markdown_pics/NLP/2/5.jpg?raw=true)

从与之前实验相同的新闻数据开始，我们首先构建了基于短语的训练语料库，然后我们使用不同的超参数训练了多个 Skip-gram 模型。与之前一样，我们使用矢量维度 300 和上下文大小 5。此设置已经在短语数据集上实现了良好的性能，并且允许我们快速比较 Negative Sampling 和 Hierarchical Softmax，无论是否使用 frequent tokens 的 subsampling。结果总结在表3 中。  

结果表明，虽然负采样在 k = 5 时仍能达到可观的精度，但使用 k = 15 可以获得相当好的性能。令人惊讶的是，虽然我们发现 Hierarchical Softmax 在没有二次采样的情况下训练时性能较低，但是当我们对频繁字进行下采样时，它成为了表现最佳的方法。这表明子采样可以导致更快的训练，并且还可以提高准确性，至少在某些情况下。  

为了最大化词组类比任务的准确性，我们通过使用大约 330 亿字的数据集来增加训练数据量。我们使用 hierarchical softmax，维度为 1000，整个句子为上下文。这导致了一个达到 72% 准确度的模型。当我们将训练数据集的大小减少到 6B 个单词时，我们的准确率降低了 66%，这表明大量的训练数据是至关重要的。  

为了进一步深入了解不同模型所表现出来的表征之间的差异，我们使用各种模型手动检查了不常见短语的最近邻居。在表4 ，我们展示了这种比较的一个样本。 与以前的结果一致，似乎短语的最佳表示是通过具有 hierarchical softmax 和二次采样的模型学习的。  

![Aaron Swartz](https://github.com/liyibo/cv_notebooks/blob/master/markdown_pics/NLP/2/6.jpg?raw=true)

## 5 Additive Compositionality  

我们证明了由 Skip-gram 模型学习的单词和短语表示呈现出一种线性结构，可以使用简单的矢量算术执行精确的类比推理。有趣的是，我们发现 Skip-gram 表示展示了另一种线性结构，它可以通过 element-wise 方式添加它们的向量来表示有意义地组合词语。表5 说明了这种现象。  

通过检查训练目标可以解释向量的加性特性。单词矢量与 softmax 非线性的输入成线性关系。当词向量被训练来预测句子中的周围词时，向量可以被看作表示词出现的上下文的分布。这些值与由输出层计算的概率以对数形式相关，所以两个单词向量的总和与两个上下文分布的乘积有关。该乘积在这里用作 AND 函数：由两个单词向量分配高概率的单词将具有高概率，而其他单词将具有低概率。因此，如果“伏尔加河”与“俄罗斯”和“河流”一起经常出现在同一个句子中，这两个单词向量的总和将导致这样一个特征向量接近“伏尔加河”向量，。  

## 6 Comparison to Published Word Representations  

![Aaron Swartz](https://github.com/liyibo/cv_notebooks/blob/master/markdown_pics/NLP/2/7.jpg?raw=true)

许多以前从事基于神经网络词汇表征的作者已经发表了他们的结果模型供进一步的使用和比较：其中最着名的作者是 Collobert 和 Weston [2]，Turian ，Mnih 和 Hinton [10]。我们从 web3 下载了他们的单词向量。Mikolov 等人[8]已经评估了单词类比任务中的这些单词表示，其中 Skip-gram 模型以巨大的优势实现了最佳性能。  

为了更深入地了解学习向量质量的差异，我们通过显示表6 中不常用词的最近邻居来提供经验性比较。这些例子表明，在大型语料库上训练的 Skip-gram 模型明显优于所有其他模型。这部分可以归因于这个模型已经被训练了大约 300 亿个单词，这比以前的工作中使用的典型大小多出大约两到三个数量级的数据。有趣的是，尽管训练集更大，但 Skip-gram 模型的训练时间仅仅是以前模型架构所需的时间复杂度的一小部分。  

## 7 Conclusion  

这项工作有几个关键贡献。我们展示了如何使用 Skip-gram 模型来训练单词和短语的分布式表示，并且证明这些表示呈现出可以进行精确的类比推理的线性结构。本文介绍的技术也可用于训练[8]中介绍的 bag-of-words 模型。  

由于采用了计算效率高的模型架构，我们成功地对比以前发布的模型多数量级的数据进行了模型训练。这导致学习的单词和短语表示的质量大大提高，特别是对于罕见的实体。我们还发现，频繁词汇的二次抽样导致更快的训练和更好地表达不常见的单词。我们的论文的另一个贡献是负采样算法，它是一种非常简单的训练方法，可以学习精确的表示，特别是对于频繁的词。  

训练算法和超参数选择的选择是一个任务特定的决定，因为我们发现不同的问题有不同的最优超参数配置。在我们的实验中，影响性能的最关键的决定是模型架构的选择，矢量的大小，子采样率以及训练窗口的大小。  

这项工作的一个非常有趣的结果是单词向量可以通过简单的向量添加而有意义地组合。学习本文中表达的短语的另一种方法是简单地用单个标记表示短语。 这两种方法的组合提供了一种强大而简单的方法，即如何表示更长的文本片段，同时具有最小的计算复杂度。因此，我们的工作可以被看作是试图用递归矩阵向量操作来表示短语的现有方法的补充[16]。  

我们根据本文描述的技术制定了用于训练词和短语向量的代码，可作为开源项目使用。

# Word2Vec Tutorial  

本教程介绍 Word2Vec 的 skip-gram 神经网络架构。本教程的目的是跳过关于 Word2Vec 的常见介绍性和抽象的见解，并深入了解更多细节。具体来说，我正在 diving into 神经网络模型。  

## The Model  

skip-gram 神经网络模型其最基本的形式实际上是非常简单的。

让我们从关于我们要做什么的高层次洞察开始。Word2Vec 使用了一个你可能在机器学习中看到过的技巧。我们将训练一个带有单个隐藏层的简单神经网络来完成特定的任务，但是我们实际上并没有将这个神经网络用于我们训练的任务！相反，目标实际上只是学习隐藏层的权重 - 我们会看到这些权重实际上是我们试图学习的“单词向量”。  

另一个你可能见过这个技巧的地方是无监督的特征学习，在这里你训练一个自动编码器来压缩隐藏层中的输入向量，并将其解压缩回输出层的原始数据。 训练完成后，您会剥离输出层（解压缩步骤），然后使用隐藏层 - 这是学习良好图像特征而不标记训练数据的诀窍。

## The Fake Task  

所以现在我们需要谈论这个我们将要建立神经网络来执行的“假”任务，然后我们会稍后再回到这个间接给我们提供那些我们真正需要单词向量的方法。

我们将训练神经网络来完成以下工作。给定一个句子中的特定单词（输入单词），查看附近的单词并随机选择一个单词。网络将告诉我们在我们的词汇中每个词是我们选择的“附近词”的可能性。

当我说“附近”时，算法实际上有一个“窗口大小”参数。一个典型的窗口大小可能是 5，意味着前后各有 5 个字（总共10个字）。
输出概率将与在我们的输入词附近找到每个词汇单词的可能性有关。例如，如果你给训练好的网络输入单词“苏联”，输出概率对于“联盟”和“俄罗斯”这样的单词来说要比“西瓜”和“袋鼠”这样无关的单词要高得多。

我们将通过输入我们的训练文档中找到的单词对来训练神经网络。下面的例子显示了一些训练样本（单词对），我们将从“The quick brown fox jumps over the lazy dog”一句中取出。我已经使用了一个 2 的小窗口大小作为例子。以蓝色突出显示的单词是输入单词。  

![Aaron Swartz](https://github.com/liyibo/cv_notebooks/blob/master/markdown_pics/NLP/3/1.jpg?raw=true)

网络将从每次配对显示的次数中学习统计数据。例如，网络可能会获得更多（“苏联”，“联盟”）的训练样本，而不是（“苏联”，“萨斯奎奇”）的训练样本。训练结束后，如果您输入“苏联”一词作为输入，那么它将输出“联盟”或“俄罗斯”的概率高于“Sasquatch”的概率。  

## Model Details  

那么这是如何表现的？

首先，你知道你不能将一个单词作为一个文本字符串输入到神经网络，所以我们需要一种方法来将这些单词表达给网络。为此，我们首先根据训练文档建立一个单词词汇表 - 假设我们有一个包含 10,000 个独特词汇的词汇表。

我们将代表像“蚂蚁”这样的输入词作为一个 one-hot vector。这个向量将有 10,000 个 components，我们将在对应于单词“蚂蚁”的位置放置一个“1”，并在所有其他位置放置一个“0”。

网络的输出是一个单一的向量（也有 10,000 个 components），对于我们的词汇表中的每个单词，它表示该单词是输入单词附近词的概率。

这是我们神经网络的架构。  

![Aaron Swartz](https://github.com/liyibo/cv_notebooks/blob/master/markdown_pics/NLP/3/2.jpg?raw=true)

隐层神经元没有激活函数，但输出神经元使用 softmax。我们稍后再回来。

当在单词对上训练这个网络时，输入是代表输入词的 one-hot 向量，并且训练输出也是代表输出词的 one-hot 向量。但是，当你在输入字上评估训练好的网络时，输出向量实际上是一个概率分布（即一组浮点值，而不是一个 one-hot 向量）。  

## The Hidden Layer  

对于我们的例子，我们要说我们正在学习 300 个特征的单词向量。因此，隐藏层将由一个 10,000 行（我们词汇表中的每个词）和 300 列（每个隐藏的神经元）的权重矩阵表示。

300 特征是 Google 在他们发布的 Google 新闻数据集训练模型中使用的特征数量。特征的数量是一个“超参数”，你只需要调整你的应用程序（即尝试不同的值，看看什么会产生最好的结果）。

如果你看这个权重矩阵的行，这些实际上是我们的词向量！  

![Aaron Swartz](https://github.com/liyibo/cv_notebooks/blob/master/markdown_pics/NLP/3/3.jpg?raw=true)

因此，所有这些的最终目标只是为了学习这个隐藏层的权重矩阵 - 当我们完成时我们会抛弃输出层！

但是，让我们回过头来看看我们要训练的这个模型的定义。

现在，你可能会问自己 - “这个 one-hot 向量几乎全是零...那有什么效果呢？”如果你将一个 1×10,000 one-hot 向量乘以一个 10,000×300 的矩阵，它将有效地选择矩阵行对应于“1”。这里有一个小例子给你一个视觉效果。  

![Aaron Swartz](https://github.com/liyibo/cv_notebooks/blob/master/markdown_pics/NLP/3/4.jpg?raw=true)

这意味着此模型的隐藏层实际上只是作为查找表来操作。隐藏层的输出只是输入单词的“单词向量”。  

## The Output Layer  

“蚂蚁”的 1×300 word 向量然后被送到输出层。输出层是一个 softmax 回归分类器。这里有一个关于 Softmax 回归的深入教程，但其要点是每个输出神经元将产生一个介于 0 和 1 之间的输出，并且所有这些输出值的总和将加起来是 1。

具体来说，每个输出神经元都有一个权重向量，它与来自隐藏层的单词向量相乘，然后将函数 exp(x) 应用于结果。最后，为了使输出总和为 1，我们将这个结果除以所有 10,000 个输出节点的结果之和。

下面是计算单词“car”的输出神经元输出的图示。  

![Aaron Swartz](https://github.com/liyibo/cv_notebooks/blob/master/markdown_pics/NLP/3/5.jpg?raw=true)

请注意，神经网络不知道输出词相对于输入词的偏移量。它并没有为输入之前的单词与之后的单词学习不同的一组概率。为了理解这个含义，假设在我们的训练语料库中，单词 'York' 的每一个单词前面都有 'New' 单词。也就是说，至少根据训练数据，'New' 将在 'York' 附近有 100% 的概率。然而，如果我们把 'York' 附近的 10 个单词随机选取其中的一个，那么 'New' 的概率不是100%。你可能选择了附近的其他单词之一。  

## Intuition  

好的，你准备好了解这个网络的一些情况吗？

如果两个不同的单词具有非常相似的“contexts”（也就是说，他们周围可能出现哪些单词），那么我们的模型需要为这两个单词输出非常相似的结果。 对于这两个词，网络输出类似上下文预测的一种方式是如果词向量相似。所以，如果两个词有相似的上下文，那么我们的网络就有动机学习这两个词的相似词向量！

两个词有相似的 contexts 是什么意思？ 我认为你可以期望像“智能”和“聪明”这样的同义词具有非常相似的 contexts。或者，与“引擎”和“传播”相关的词语可能也有类似的语境。

这也可以处理词干 - 网络可能会学习单词“ant”和“ants”的相似单词向量，因为它们应该有类似的上下文。  

## part 2

在 word2vec 教程的第 2 部分中，我将介绍对基本 skip-gram 模型的一些额外修改，实际上这些修改对于使其可以有效训练非常重要。

当您阅读 Word2Vec 的 skip-gram 模型教程时，您可能已经注意到了一些东西 - 这是一个巨大的神经网络！

在我给出的例子中，我们有 300 个 components 的单词向量和 10,000 个单词的词汇表。回想一下，神经网络有两个权重矩阵 - 隐藏层和输出层。这两层都会有一个权重矩阵，每个权重矩阵为 300 x 10,000 = 300 万！

在一个大的神经网络上运行梯度下降是很缓慢的。更糟糕的是，你需要大量的训练数据才能调整许多权重并避免过拟合。数百万的权重乘以数十亿训练样本意味着训练这个模型将成为一个野兽。

Word2Vec 的作者在他们的第二篇论文中讨论了这些问题。

第二篇论文有三个创新点：

- 1 在他们的模型中将常用单词对或短语视为单个“单词”。  

- 2 对频繁的词进行子采样以减少训练实例的数量。  

- 3 用他们称之为“负采样”的技术修改优化目标，这会使每个训练样本只更新一小部分模型的权重。

值得注意的是，对频繁词进行二次抽样和应用负抽样不仅减少了训练过程的计算负担，而且还提高了其结果词向量的质量。

## Word Pairs and “Phrases”  

作者指出，像“Boston Globe”（一家报纸）这样的词组与“Boston”和“Globe”这两个单词有着非常不同的含义。因此，将“Boston Globe”无论出现在文本中的任何地方都视为具有自己词汇向量表示的单个词汇，这是有道理的。

您可以在他们发布的模型中查看结果，该结果从 Google 新闻数据集中对 1000 亿字进行了训练。在模型中加入短语使词汇量增加到 300 万字！

如果你对他们所产生的词汇感兴趣，我可以稍微探讨一下。你也可以在这里浏览他们的词汇。

短语检测包含在他们的论文的“Learning Phrases”部分。他们在 word2phrase.c 中分享了他们的实现。

我不认为他们的短语检测方法是他们论文的一个关键贡献，但是我会分享一些，因为它非常简单。

他们的工具每次只查看 2 个单词的组合，但可以多次运行以获得更长的短语。因此，第一遍会选择“New_York”，然后再次运行它 将 “New_York_City” 选为“New_York”和“City”的组合。

该工具计算训练文本中两个单词的每个组合出现的次数，然后将这些计数用于等式中以确定将哪些单词组合成短语。它还喜欢使用不常见字组成的短语词组，以避免 “and the” 或“this is”等常见词组成词组。

我认为有一种替代词组识别策略，就是将所有维基百科文章的标题用作词汇。

## Subsampling Frequent Words  

在本教程的第1 部分中，我展示了如何从源文本创建训练样本，但我会在此重复。下面的例子显示了一些训练样本（单词对），我们将从“The quick brown fox jumps over the lazy dog”一句中取出。我已经使用了一个 2 的小窗口作为例子。以蓝色突出显示的单词是输入单词。  

![Aaron Swartz](https://github.com/liyibo/cv_notebooks/blob/master/markdown_pics/NLP/3/1.jpg?raw=true)

用“the”这个常见词有两个“问题”：

- 1 当看单词对时，(“fox”，“the”)并没有告诉我们很多关于“fox”的含义。“the”几乎出现在每个单词的上下文中。  

- 2 我们将有比我们需要为“the”学习一个好的向量的更多的样本(“the”，...)。  

Word2Vec 实现了一个“subsampling”方案来解决这个问题。对于我们在训练文本中遇到的每个单词，我们都有可能将其有效的从文本中删除。删除单词的概率与单词的频率有关。

如果我们的窗口大小为 10，并且从文本中删除“the”的特定实例：

- 1 当我们训练剩余的单词时，“the”不会出现在任何上下文窗口中。  

- 2 当“the”是输入词时，我们将减少 10 个训练样本。

请注意这两个效果如何帮助解决上述两个问题。

## Sampling rate  

word2vec C 代码实现了计算词汇表中给定单词的概率的等式。

wi 是单词，z（wi）是语料库中所有单词的分数。例如，如果单词“peanut”在 10 亿字的语料库中出现 1000 次，则 z（'peanut'）= 1E-6。

代码中还有一个 'sample' 参数，用于控制发生多少次采样，默认值为 0.001。较小的“sample”意味着 words 更难以保留。

P（wi）是保留单词的概率：

$P(w_i) = (\sqrt {\frac {z(w_i)}{0.001}} + 1) \cdot \frac {0.001}{z(w_i)}$

您可以在Google中快速绘制此图形以查看形状。  

没有一个单词应该是语料库中非常大的比例，所以我们希望在 x 轴上查看相当小的值。

这里有一些有趣的地方在这个函数（也是使用默认的样本值0.001）。

- 当 z（wi）≤ 0.0026 时，P（wi）= 1.0（保持100%的概率）。这意味着只有占总词数超过 0.26% 的词将被二次抽样。  

- 当 z（wi）= 0.00746 时，P（wi）= 0.5（保持50%的机会）。  

- 当z（wi）= 1.0，P（wi）= 0.033（保持3.3%的机会）。也就是说，如果语料库完全由 wi 词组成，这当然是荒谬的。  

你可能会注意到这篇论文定义的这个函数与 C 代码中实现的有些不同，但我认为 C 实现是更权威的版本。  

## Negative Sampling  

训练一个神经网络意味着输入一个训练样本就稍微调整所有的神经元权重，以便更准确地预测训练样本。换句话说，每个训练样本都会调整神经网络中的所有权重。

正如我们上面所讨论的那样，我们的单词词汇的大小意味着我们的 skip-gram 神经网络有很多权重，所有这些都将被我们数十亿训练样本中的每一个稍微更新！

负采样通过让每个训练样本只修改一小部分权重而不是全部权重来解决这个问题。这是它的工作原理。

当在单词对(“fox”, “quick”)上训练网络时，请回想一下，网络的“标签”或“正确输出”是一个 one-hot 向量。也就是说，对于对应于“quick”的输出神经元输出 1，并且对于所有其他数千个输出神经元输出 0。

对于 negative  抽样，我们将随机选择一小部分“negative ”单词（比如 5）来更新权重。（在这种情况下，“negative ”一词是我们希望网络输出 0 的那个词）。我们还将更新我们的“positive”单词的权重（在我们当前的例子中，这是“quick”一词）。

该论文指出，选择 5-20 个单词适用于较小的数据集，并且对于大型数据集只能使用 2-5 个单词。  

回想一下，我们模型的输出层有一个 300 x 10,000 的权重矩阵。所以我们只是更新我们正面词的权重（“quick”），加上我们想要输出的其他 5 个词的权重。这总共有 6 个输出神经元，总共有 1,800 个权重值。这只是输出层 3M weights 的 0.06%！

在隐藏层中，只更新输入词的权重（无论您是否使用 Negative 采样，情况都是如此）。

## Selecting Negative Samples  

“负样本”（即我们将训练输出为 0 的 5 个输出字）是使用“unigram distribution”选择的。

本质上，选择一个词作为负样本的概率与其频率有关，更频繁的词更可能被选作负样本。

在 word2vec C 实现中，您可以看到这个概率的等式。每个单词的权重等于其频率（字频）的 3/4 次方。选择一个词的概率只是它的权重除以所有单词的权重之和。

$P(w_i) = \frac {f(w_i)^{3/4}}{\sum_{j=0}^{n}(f(w_i)^{3/4})}$

将频率提高到 3/4 power 的决定似乎是经验性的；在他们的论文中他们说它超越了其他数字。您可以查看函数的形状 - 只需在Google中输入：“plot y = x^(3/4) and y = x”，然后放大范围 x = [0，1]。它有一个略微增加的曲线。

这种选择在 C 代码中实现的方式很有趣。他们有一个拥有 100M 元素的大array （他们称之为 unigram table）。他们多次使用用词汇表中每个单词的索引填充这个表格，单词索引出现在表格中的次数由 $P(w_i) * table_size$ 给出。然后，为了实际选择一个负样本，您只需生成一个介于 0 到 100M 之间的随机整数，然后在表格中使用该索引处的单词。由于较高的概率词在表格中出现次数更多，因此您更有可能选择这些词语。  

# Enriching Word Vectors with Subword Information  

## Abstract  

在大型未标记语料库上训练的连续词表示对许多自然语言处理任务都很有用。学习这种表示的流行模型通过为每个单词分配一个不同的矢量来忽略单词的形态。这是一个限制，特别是对于词汇量很大且词汇很多的语言。在本文中，我们提出了一种基于 skip-gram 模型的新方法，其中每个单词表示为a
bag of character n-grams。矢量表示与每个字符 n-gram 相关联；单词被表示为这些表示的总和。我们的方法速度很快，可以快速训练大型语料库中的模型，并允许我们计算未出现在训练数据中的单词的单词表示。我们用九种不同的语言来评估我们的词汇表达，包括词汇相似性和类比任务。通过与最近提出的形态词表示进行比较，我们证明了我们的向量在这些任务上达到了最先进的性能。  

## 1 Introduction  

学习连续的词汇表达在自然语言处理方面有着悠久的历史。这些表示通常从大型未标记语料中使用共现统计计算出来。称为分布式语义学的大量工作，研究了这些方法的性质。在神经网络社区中，Collobert 和 Weston（2008）提出使用前馈神经网络学习单词嵌入，通过基于左侧两个单词和右侧两个单词预测单词。最近，米科洛夫等人。提出了简单的对数双线性模型来有效地学习非常大的语料库上的词的连续表示。  

这些技术中的大多数通过不同的向量表示词汇表中的每个词，而不用参数共享。尤其是，他们忽略了单词的内部结构，这是形态丰富的语言（如土耳其语或芬兰语）的重要限制。例如，在法语或西班牙语中，大多数动词有超过四十种不同的变形形式，而芬兰语则有十五种名词形式。这些语言包含许多在训练语料库中很少发生（或根本不发生）的单词形式，这使得难以学习好的单词表示。由于许多单词遵循规则形成，因此可以通过使用字符级别信息来改进形态丰富的语言的向量表示。  

在本文中，我们建议学习字符 n-gram 的表示，并将单词表示为 n-gram 向量的总和。我们的主要贡献是引入连续 skip-gram 模型的扩展，其中考虑了子字词信息。我们用九种不同形态的语言评估这个模型，显示出我们方法的好处。  

## 2 Related work  

**Morphological word representations** 近年来，已经提出了许多方法将形态信息结合到词表示中。为了更好地模拟稀有词汇，Alexandrescu 和Kirchhoff（2006）引入了分解神经语言模型，其中词语表示为特征集合。这些特征可能包括形态信息，并且该技术已成功应用于形态丰富的语言，如土耳其语（Sak等，2010）。最近，一些作品提出了不同的组合函数来从语素中推导词的表示。这些不同的方法依赖于词的形态分解，而我们的则不是。同样，Chen 等人介绍了一种联合学习汉字和字符嵌入的方法。Cui 等人提出将形态上相似的词限制为具有相似的表示。Soricut 和 Och（2015）描述了一种学习形态变换的向量表示的方法，允许通过应用这些规则来获取未看见的单词的表示。Cotterell 和 Schütze（2015年）介绍了形态学注释数据的词汇表征。最接近我们的方法，Schütze（1993）通过奇异值分解学习了 four-grams 字符的表示，并且通过对 four-grams 表示求和而得出单词的表示。最近，Wieting等人（2016）也提出使用字符 n-gram 计数向量表示单词。然而，用于学习这些表示的目标函数基于释义对，而我们的模型可以在任何文本语料库上进行训练。  

**Character level features for NLP** 与我们的工作密切相关的另一个研究领域是自然语言处理的字符级模型。这些模型将单词分割成字符并旨在直接从字符学习语言表示。第一类这样的模型是递归神经网络，应用于语言建模，文本规范化，词性标注和解析。另一个模型家族是卷积神经网络训练的字符，应用于词性标注，情感分析，文本分类和语言建模。Sperr 等人引入了一种基于受限玻尔兹曼机器的语言模型，其中词语被编码为一组字符节点。最后，最近在机器翻译方面的工作已经提出使用子字词单元来获得罕见字词的表示。  

## 3 Model  

在本节中，我们建议我们的模型在考虑形态学的同时学习单词表示。我们通过考虑子字单位来对形态进行建模，并且用字符 n-gram 的总和来表示字。我们将首先介绍我们用来训练单词向量的一般框架，然后展示我们的子词模型并最终描述我们如何处理字符 n-gram 的词典。  

### 3.1 General model  

我们首先简要回顾一下 Mikolov 等人提出的 skip-gram 模型。我们的模型来源于此。给定一个大小为 W 的单词词汇表，其中一个单词由其索引 $w \in \{1,...,W\}$ 来标识，目标是学习每个单词 w 的向量表示。受分布假设（Harris，1954）的启发，词汇表征被训练以预测出现在其上下文中的词汇。更正式地说，给定一个大的训练语料库，skip-gram 模型的目标是最大化以下对数似然函数：  

$\sum_{t=1}^{T} \sum_{c \in C_t} \log {p(w_c|w+t)}$

其中上下文 Ct 是围绕单词 wt 的词的索引的集合。使用上述单词向量来参数化给定 wt 的上下文词 wc 的概率。现在，让我们考虑一下，我们得到了一个评分函数 s，它将 (word, context)对映射为分数。  

定义上下文词的概率的一个可能选择是 softmax：  

$p(w_c|w_t) = \frac {e^{s(w_t,w_c)}}{\sum_{j=1}^W e^{s(w_t,j)}}$

然而，这样的模型并不适用于我们的情况，因为它暗示，给定一个单词 wt，我们只能预测一个上下文单词 wc。  

预测上下文单词的问题可以被定义为一组独立的二进制分类任务。然后，目标是独立预测上下文单词的存在（或不存在）。对于位置 t 处的单词，我们将所有上下文单词视为正例，并从字典中随机 sample negatives。对于选择的上下文位置 c，使用二元逻辑损失，我们获得以下负对数似然性：  

$\log{(1+e^{-s(w_t,w_c)})} + \sum_{n \in N_{t,c} \log {(1+e^{s(w_t,n)})}}$

其中 $N_{t,c}$ 是从词汇中抽取的一组反例。通过表示逻辑损失函数：$l: x \rightarrow \log{1+e^{-x}}$，我们可以将目标重新写为：  

$\sum_{t=1}^T \left[ \sum_{c \in C_t} l(s(w_t,w_c)) + \sum_{n \in N_{t,c}} l(-s(w_t,n)) \right]$

词 wt 和上下文词 wc 之间的评分函数使用词向量参数化。让我们为词汇中的每个单词 w 定义 $R^d$ 中的两个向量 $u_w$ 和 $v_w$。这两个向量在文献中有时被称为输入和输出向量。特别是，我们有向量 $u_{wt}$ 和 $v_{wc}$，分别对应于 wt 和 wc。然后可以将单词和上下文向量之间的标量积作为得分。本节描述的模型是 Mikolov 等人提出的负样本抽样模型。  

### 3.2 Subword model  

通过为每个单词使用不同的矢量表示形式，skip-gram 模型将忽略单词的内部结构。在本节中，我们提出了一个不同的评分函数 s，以便考虑到这些信息。  

每个单词 w 表示为 a bag of character n-gram。我们在单词的开头和结尾添加特殊的边界符号 $<$ 和 $>$，以便从其他字符序列中区分前缀和后缀。 我们还将 w 本身包含在它的 n 元组中，以学习每个单词的表示（in addition to character n-grams）。以 where 和 n = 3 这个词为例，它将由字母 n-grams 表示：  

$<wh, whe, her, ere, re>$

以及特殊的序列  

$<where>$

请注意，对应于 $<her>$ 这个词的序列不同于 where 中的 tri-gram her。在实践中，我们提取 n 大于或等于3，小于或等于 6 的所有 n 元组。这是一个非常简单的方法，可以考虑不同的 n 元组，例如采用所有的前缀和后缀。  

假设给你一个大小为 G 的 ngram 字典，给定一个单词 w，让我们用 $g_w \subset \{1,...,G\}$ 表示出现在 w 中的一组 n-gram。我们对 n-gram 中每个 g 都分配一个 $z_g$。通过其 n-gram 的向量表示的总和来表示一个词。我们因此获得评分函数：  

$s(w,c) = \sum_{g \in g_w} z^T_g v_c$

这个简单的模型允许跨越单词共享表示，从而允许学习罕见单词的可靠表示。  

为了限制我们模型的内存需求，我们使用哈希函数将 n-gram 映射为 1 到 K 中的整数。我们使用 Fowler-Noll-Vo 哈希函数。我们设置 $K = 2.10^6$以下。最终，一个单词由单词字典中的索引以及它包含的一组散列 n-gram 来表示。  

## 4 Experimental setup  

### 4.1 Baseline  

在大多数实验中，我们将模型与 word2vec2 包中 skip-gram 和 cbow 模型的 C 实现进行比较。  

### 4.2 Optimization  

我们通过对之前呈现的负对数似然性执行随机梯度下降来解决我们的优化问题。所有线程以异步方式共享参数和更新向量。  

### 4.3 Implementation details  

对于我们的模型和基线实验，我们使用以下参数：单词向量的维数为 300。对于每个正例，我们随机抽样 5 个负样本。我们使用大小为 c 的上下文窗口，并在 1 到 5 之间对大小 c 进行均匀采样。为了对最频繁的单词进行二次采样，我们使用 $10^{-4}$ 的拒绝阈值。在建立单词词典时，我们保留在训练集中出现至少五次的单词。对于 skip-gram 基线，步长 $\gamma_0$ 设置为 0.025，对于我们的模型和 cbow 基线，步长 $\gamma_0$ 设置为 0.05。这些是 word2vec 软件包中的默认值，也适用于我们的模型。  

在英文数据上使用此设置，我们的字符 n-gram 模型比 skip-gram 基线的训练慢大约 1.5 倍。事实上，我们处理 105k words/second/thread vs  基线的145k words/second/thread。我们的模型是用 C++ 实现的，并且是公开可用的。  

### 4.4 Datasets  

除了与以前的工作（5.3节）相比较外，我们还对维基百科数据进行了模型训练。我们用阿拉伯语，捷克语，德语，英语，西班牙语，法语，意大利语，罗马尼亚语和俄语下载了九种语言的维基百科转储。我们使用 Matt Mahoney 的预处理 perl 脚本对原始维基百科数据进行了规范化。所有数据集都进行了混洗，并且我们通过对它们进行了五遍训练来模拟我们的模型。  

## 5 Results  

我们在五个实验中评估我们的模型：评估单词相似性和单词类比，与最先进的方法进行比较，分析训练数据大小以及字符 n-gram 的大小的影响。我们将在下面的章节中详细介绍这些实验。  

### 5.1 Human similarity judgement  

![Aaron Swartz](https://github.com/liyibo/cv_notebooks/blob/master/markdown_pics/NLP/4/1.jpg?raw=true)

我们首先评估我们的表示在词语相似性/相关性任务上的质量。我们通过计算 Spearman 的等级相关系数来判断人类判断与矢量表示之间的余弦相似度。对于德语，我们比较了三个数据集上的不同模型：GUR65，GUR350 和 ZG222。对于英语，我们使用 Finkelstein 等人介绍的 WS353 数据集，以及 Luong 等人提出的稀有词数据集（RW）。我们评估翻译数据集 RG65 上的法语单词向量。使用（Hassan和Mihalcea，2009）中描述的数据集评估西班牙语，阿拉伯语和罗马尼亚语单词向量。使用 Panchenko 等人介绍的 HJ 数据集评估俄语单词向量。  

我们在表1 报告了我们的方法和基线方法在所有数据集的结果。这些数据集中的一些单词没有出现在我们的训练数据中，因此我们无法使用 cbow 和skip-gram 基线获取这些单词的单词表示。为了提供可比较的结果，我们建议默认使用这些词的空向量。由于我们的模型利用了子词信息，因此我们还可以计算词外单词的有效表示。我们通过取其 n-gram 向量的总和来做到这一点。当使用空向量表示 OOV 词时，我们将我们的方法称为 sisg- 和 sisg。  

首先，通过查看表1，我们注意到使用子字信息的所提出的模型（sisg）优于除了英语 WS353 数据集之外的所有数据集的基线。此外，词汇外单词（sisg）的计算向量总是至少与不这样做（sisg-）一样好。这证明了以字符 n-gram 形式使用子字信息的优点。  

其次，我们观察到使用字符 n-gram 对阿拉伯语，德语和俄语的影响比英语，法语或西班牙语更重要。德语和俄语展示语法变体，其中德语四种，俄语六种。另外，很多德语单词都是复合词; 例如，名词短语“table tennis”用一个词语写成“Tischtennis”。通过利用“Tischtennis”和“Tennis”之间的字符级相似性，我们的模型不会将这两个单词表示为完全不同的单词。  

最后，我们观察到在 English Rare Words 数据集（RW）上，对于 OOV 词我们的方法优于基线。这是因为英语 WS353 数据集中的单词是不使用子字词信息就可以获得良好矢量的常用单词。当评估不太常见的单词时，我们发现在单词之间的字符级别使用相似性可以帮助学习良好的单词向量。  

### 5.2 Word analogy tasks  

![Aaron Swartz](https://github.com/liyibo/cv_notebooks/blob/master/markdown_pics/NLP/4/2.jpg?raw=true)

我们现在评估我们的词类比问题的方法，形式是 A is to B as C is to D，其中 D 必须由模型预测。我们使用 Mikolov 等人介绍的数据集。分别有捷克语，英语，德语，意大利语。有些问题包含了我们训练语料库中没有出现的词汇，因此我们从评估中排除了这些问题。  

我们在表2 中报告了不同模型的准确性。我们观察到形态信息显着改善了语法任务；我们的方法胜过基线。相反，它不利于语义问题，甚至会降低德语和意大利语的表现。请注意，这与我们考虑的字符 n-gram 长度的选择密切相关。我们在 5.5 节展示，当 n-gram 的大小被选择为最佳时，语义类比降低得更少。另一个有趣的发现是，正如预期的那样，对形态丰富的语言如捷克语和德语改进的更多。  

### 5.3 Comparison with morphological representations  

![Aaron Swartz](https://github.com/liyibo/cv_notebooks/blob/master/markdown_pics/NLP/4/3.jpg?raw=true)

我们还将我们的方法与以前关于词相似任务的工作进行比较。使用的方法是：Luong 等人的递归神经网络，Qiu 等人的 morpheme cbow，以及 Soricut 和Och 的形态转换。为了使结果具有可比性，我们使用与我们比较的方法相同的数据集对我们的模型进行了训练。我们还将我们的方法与 Botha 和 Blunsom 引入的对数双线性语言模型进行了比较，该模型在 Europarl 和新闻评论语料库上进行了训练。再次，我们使用相同的数据对我们的模型进行了训练，以使结果可比。使用我们的模型，我们通过对字符 n-gram 的表示进行求和来获得词外词的表示。我们在表3 中报告了结果。我们观察到，与基于从形态 segmentors 获得的子词信息的技术相比，我们的简单方法表现良好。我们还观察到，我们的方法优于基于前缀和后缀分析的 Soricut 和 Och 的方法。德语的巨大改进是由于他们的方法不能模拟名词复合，与我们的相反。  

### 5.4 Effect of the size of the training data  

![Aaron Swartz](https://github.com/liyibo/cv_notebooks/blob/master/markdown_pics/NLP/4/4.jpg?raw=true)

由于我们利用单词之间的字符级相似性，我们能够更好地模拟不常用的单词。因此，我们也应该对我们使用的训练数据的大小更加健壮。为了评估这一点，我们建议训练集大小作为函数来评估我们的词向量在相似任务上的效果。我们在图1 中报告结果。  

正如在 5.1 节中介绍的实验一样，并非评估集的所有单词都出现在维基百科数据中。同样，默认情况下，我们对这些单词使用空矢量（sisg-）或通过对 n-gram 表示（sisg）进行求和来计算矢量。随着数据集的收缩，词外率越来越高，因此 sisg- 和 cbow的性能必然下降。然而，所提出的模型（sisg）将 non-trivial 的向量分配给以前未见过的单词。  

首先，我们注意到对于所有数据集和所有大小，所提议的方法（sisg）表现比基线好。然而，随着越来越多的数据可用，基线 cbow 模型的性能会变得更好。另一方面，我们的模型似乎很快饱和，增加更多的数据并不总是会导致改进的结果。  

其次，最重要的是，我们注意到，即使使用非常小的训练数据集，所提出的方法也提供了非常好的单词向量。例如，在德国的 GUR350 数据集上，我们的模型（sisg）训练了 5% 的数据比完整数据集训练的 cbow 基线获得了更好的性能。另一方面，在英国 RW 数据集上，使用维基百科语料库的 1%，我们得到45 的相关系数，这比在整个数据集上训练的 cbow 的性能要好。这具有非常重要的实际意义：可以在有限大小的数据集上计算性能良好的单词向量，并且仍能很好地处理以前未见过的单词。通常，在特定应用程序中使用 vectorial 字表示时，建议对与应用程序相关的文本数据重新训练模型。  

### 5.5 Effect of the size of n-grams  

![Aaron Swartz](https://github.com/liyibo/cv_notebooks/blob/master/markdown_pics/NLP/4/5.jpg?raw=true)

所提出的模型依赖于使用字符 ngram 来表示单词作为向量。正如在 3.2 节中提到的。我们决定使用 3 到 6 个字符的 n-gram。这种选择是任意的，因为这些长度的 n-gram 将涵盖范围广泛的信息。它们将包括短后缀以及更长的根。在这个实验中，我们凭经验检查了 n-gram范围对性能的影响。我们在表4 中报告英语和德语关于词相似性和类比数据集的结果。  

我们观察到，对于英语和德语，我们任意选择 3-6 是一个合理的决定，因为它提供了令人满意的跨语言表现。长度范围的最佳选择取决于所考虑的任务和语言，应适当调整。但是，由于测试数据的稀缺性，我们没有实施任何适当的验证程序来自动选择最佳参数。尽管如此，如 3-6 等较大范围提供了合理数量的子词信息。  

该实验还表明，包括长 n-gram 是很重要的。对于德语来说尤其如此，因为许多名词都是由几个单位组成的 compounds，只能通过较长的字符序列才能被捕获。在类比任务上，我们观察到使用更大的 n-gram 有助于语义类比。然而，结果总是通过取 n≥3 而不是 n≥2 来改善，这表明字符 2-gram 对于该任务没有提供信息。如3.2节所述，在计算字符 n-gram 之前，我们添加和附加特殊的位置字符来表示字的开始和结束。  

### 5.6 Language modeling  

![Aaron Swartz](https://github.com/liyibo/cv_notebooks/blob/master/markdown_pics/NLP/4/6.jpg?raw=true)

在本节中，我们描述了在语言建模任务中使用我们的方法获得的单词向量的评估。我们使用 Botha 和 Blunsom（2014）介绍的数据集，以五种语言（CS，DE，ES，FR，RU）评估我们的语言模型。每个数据集大约包含一百万个训练 tokens，我们使用与 Botha 和 Blunsom（2014）相同的预处理和数据拆分。  

我们的模型是一个具有 650 个 LSTM 单元的递归神经网络，regularized with dropout（概率为0.5）和 weight decay（正则化参数为10-5）。我们使用Adagrad 算法学习参数，学习率为 0.1，剪切大于1.0范数的梯度。 我们在 [-0.05,0.05] 范围内初始化网络的权重，并使用 20 的批处理大小。考虑两个基线：我们将我们的方法与 Botha 和 Blunsom（2014）的对数双线性语言模型以及 Kim 等人的字符感知语言模型相比。我们在语言建模任务的训练集上训练了带有字符 n-gram 的单词向量，并使用它们来初始化我们的语言模型的查找表。我们报告我们的模型的测试困惑，而不使用预先训练的单词向量（LSTM），预先训练没有子词信息（sg）和我们的向量（sisg）的单词向量。结果列于表5。  

我们观察到，使用预先训练的词表示初始化语言模型的查找表可以提高基线 LSTM 上的测试困惑度。最重要的观察结果是使用用子字信息训练的字表示胜过普通的 skip-gram 模型。  

## 6 Qualitative analysis  

### 6.1 Nearest neighbors  

我们在表7 中报告样本定性结果。对于选定的单词，我们根据余弦相似性显示使用所提议方法训练的矢量和 skipgram 基线的最近邻居。正如预期的那样，使用我们的方法的复杂，技术和不常用词语的最近邻居比使用基线模型获得的更好。  

### 6.2 Character n-grams and morphemes  

![Aaron Swartz](https://github.com/liyibo/cv_notebooks/blob/master/markdown_pics/NLP/4/7.jpg?raw=true)

我们想定性评估一个单词中最重要的 n 元组是否对应于 morphemes。为此，我们将一个单词向量构造成 n-gram 的总和。如 3.2 节所述，每个单词 w 表示为它的 n 元组的总和。  

我们在表6 中以三种语言显示选定词的 ranked n-gram。  

对于有很多复合名词的德语，我们观察到最重要的 n-gram 对应于有效的词素。好例子包括 Autofahrer（汽车司机），其中最重要的 n-gram 是Auto（汽车）和 Fahrer（司机）。  

### 6.3 Word similarity for OOV words  

如 3.2 节所述，我们的模型能够为未出现在训练集中的单词构建单词向量。对于这样的词，我们简单地平均它的 n 元组的向量表示。为了评估这些表示的质量，我们通过从英语 RW 相似性数据集中选择几个单词对来分析哪些 n-gram 匹配最适合OOV单词。对于每一对单词，我们显示出现在单词中的每对 n-gram 之间的余弦相似度。为了模拟具有更多 OOV 字的设置，我们使用在维基百科数据的 1% 上训练的模型。结果如图2 所示。  

我们观察有趣的模式，显示子词匹配正确。事实上，对于 chip 这个词，我们清楚地看到在 microcircuit 中有两组 n-gram 匹配得很好。这些大致对应于micro 和 circuit，并且中间的 n-gram 不匹配。  

### 7 Conclusion  

在本文中，我们调查了一种简单的方法，通过考虑子词信息来学习单词表示。我们的方法将字符 n-gram 结合到 skip-gram 模型中。由于其简单性，我们的模型训练速度快，不需要任何预处理或监督。我们表明，我们的模型胜过了不考虑子词信息的基线，以及依赖于形态分析的方法。我们将开源实现我们的模型，以便于比较未来在学习子词表示方面的工作。

# Bag of Tricks for Efficient Text Classification  

## Abstract  

本文探讨了一种简单有效的文本分类基准。实验表明，我们的快速文本分类 fastText 在准确性方面通常与深度学习分类器相当，在训练和评估方面速度快很多。我们可以在不到 10 分钟的时间内使用标准的多核 CPU 对超过 10 亿个单词进行快速文本训练，并在不到一分钟的时间内对 312K 类中的 50 万个句子进行分类。  

## 1 Introduction  

文本分类是自然语言处理中的一项重要任务，具有许多应用，如网络搜索，信息检索，排序和文档分类。最近，基于神经网络的模型变得越来越流行。虽然这些模型在实践中取得了非常好的表现，但是它们在 train 和 test 时往往相对较慢，限制了它们在非常大的数据集上的使用。  

同时，线性分类器通常被认为是文本分类问题的强基线。尽管它们很简单，但如果使用了正确的特征，他们通常会获得最先进的性能。他们也有可能扩展到非常大的语料库。  

在这项工作中，我们探讨了如何在文本分类的背景下将这些基线扩展为具有大输出空间的非常大的语料库。受近期有效词表征学习的启发，我们表明具有rank 约束和快速损失近似的线性模型可以在十分钟内训练十亿字，同时实现与最先进的技术相媲美的性能。我们在两项不同任务中评估我们的方法 fastText 的质量，即标签预测和情感分析。  

## 2 Model architecture  

![Aaron Swartz](https://github.com/liyibo/cv_notebooks/blob/master/markdown_pics/NLP/5/1.jpg?raw=true)

句子分类的简单而有效的基线是将句子表示为词袋（BoW）并且训练线性分类器，例如逻辑回归或 SVM。但是，线性分类器不会在特征和类之间共享参数。 这可能会限制它们在大输出空间的情况下的泛化，其中一些类只有很少的例子。这个问题的常见解决方案是将线性分类器分解成低秩矩阵或使用多层神经网络。  

图1 显示了一个 rank 约束的简单线性模型。第一个权重矩阵 A 是单词上的查找表。然后将单词表示平均为文本表示，然后将其馈送给线性分类器。文本表示是一个可能被重用的隐藏变量。这种结构类似于 Mikolov 等人的 cbow 模型，其中中间词被标签取代。我们使用 softmax 函数 f 来计算预定义类中的概率分布。对于一组 N 个文档，这导致了对这些类的负对数似然性的最小化。  

### 2.1 Hierarchical softmax  

当类的数量很大时，计算线性分类器的计算量很大。更确切地说，计算复杂度为O(kh)，其中 k 是类的数量，h 是文本表示的维数。为了改善我们的运行时间，我们使用基于霍夫曼编码树的分层 softmax。在训练期间，计算复杂度降至 O(h log2(k))。  

当搜索最可能的类别时，分层 softmax 在测试时间也是有利的。每个节点都与从根节点到该节点的路径概率相关联。节点的概率总是低于其父节点的概率。通过深度探索树首先搜索并跟踪叶子之间的最大概率使我们能够丢弃与小概率相关的任何分支。在实践中，我们观察到在测试时间O(h log2(k))的复杂度降低。这种方法进一步扩展到以 O(log)T))为代价，使用 binary heap 计算 T-top 目标。  

### 2.2 N-gram features  
词袋是单词顺序不变的，但考虑到这个顺序通常在计算上非常昂贵。相反，我们 a bag of n-grams 作为附加特征来捕获有关词序的部分信息。这在实践中是非常有效的，同时对明确使用顺序的方法实现可比较的结果。  

我们通过使用 hashing 技巧，使用与 Mikolov 等人相同的 hashing 函数，保持了快速和高效的 n-gram 映射。如果我们只使用 bigrams 就需要 10M
bins，否则 100M。  

## 3 Experiments  

我们在两个不同的任务上评估 fastText。首先，我们将它与情感分析问题上的现有文本分类进行比较。然后，我们评估其扩展到标签预测数据集上的大输出空间的能力。请注意，我们的模型可以用 Vowpal Wabbit 库实现，但我们在实践中观察到，我们的定制实现速度至少提高了2-5倍。  

### 3.1 Sentiment analysis  

![Aaron Swartz](https://github.com/liyibo/cv_notebooks/blob/master/markdown_pics/NLP/5/2.jpg?raw=true)

**Datasets and baselines** 我们采用 Zhang 等人的 8 个数据集和评估协议。我们报告 Zhang 等人的 n-gram 和 TFIDF 基线，以及 Zhang 和LeCun的字符级卷积模型（char-CNN），Xiao 和 Cho 的基于字符的卷积循环网络（char-CRNN）和非常深的卷积网络 （VDCNN）等。  

**Results** 我们在图1 中给出了结果。我们使用 10 个隐藏单元并运行 5 个 epoch 的 fastText，并在 {0.05,0.1,0.25,0.5} 的验证集上选择了一个学习率。在这个任务中，添加 bigram 信息将使性能提高 1-4%。总的来说，我们的准确率比 char-CNN 和 char-CRNN 略好，并且比 VDCNN 差一些。请注意，我们可以通过使用更多的 n-gram 来略微提高精度，例如使用 trigrams，搜狗数据的性能可以达到 97.1%。最后，图3 显示我们的方法与 Tang 等人提出的方法具有可比性。我们调整验证集上的超参数，并观察使用高达 5 的 n-grams 可以获得最佳性能。不像 Tang 等人，fastText 不使用预先训练的词嵌入，这可以解释 1% 的准确率差异。  

![Aaron Swartz](https://github.com/liyibo/cv_notebooks/blob/master/markdown_pics/NLP/5/3.jpg?raw=true)

**Training time** char-CNN 和 VDCNN 都使用 NVIDIA Tesla K40 GPU 进行训练，而我们的模型则使用 20 个线程在 CPU 上进行训练。表2 显示使用卷积的方法比 fastText 慢几个数量级。虽然使用更新的 CUDA 实现的卷积可以使 char-CNN 的速度提高 10 倍，但 fastText 只需不到一分钟的时间就可以训练这些数据集。Tang 等人的 GRNN 方法，每个 CPU 使用一个线程大约需要 12 个小时。与基于神经网络的方法相比，我们的加速比随着数据集的大小而增加，至少达到 15,000 倍的加速。  

![Aaron Swartz](https://github.com/liyibo/cv_notebooks/blob/master/markdown_pics/NLP/5/4.jpg?raw=true)

### 3.2 Tag prediction  

![Aaron Swartz](https://github.com/liyibo/cv_notebooks/blob/master/markdown_pics/NLP/5/5.jpg?raw=true)

**Dataset and baselines** 为了测试我们方法的可伸缩性，我们对 YFCC100M 数据集进行了进一步评估，该数据集由几乎 100M 的带有标题，标题和标签的图像组成。我们专注于根据标题预测标签（我们不使用图像）。我们删除少于 100 次的字词和标签，并将数据分成 train,，验证和测试集。train 包含 91,188,648 个示例（1.5B tokens）。验证集有 930,497 个示例和测试集 543,424。词汇大小为 297,141，并且有 312,116 个独特标签。我们将发布一个脚本来重新创建这个数据集，以便我们的数据可以被复制。我们在1 报告精度。  

我们考虑基于频率的基线预测最频繁的标签。我们还与 Tagspace 进行了比较，这是一个类似于我们的标签预测模型，但是基于 Weston 等人的 Wsabie 模型，虽然使用卷积描述标签空间模型，但我们认为线性版本可以实现可比较的性能，但速度更快。  

**Results and training time** 表5 给出了 fastText 和基线的比较。我们运行 5 个 epoch 的 fastText，并将其与 Tagspace 的两种尺寸的隐藏层（即50和200）进行比较。两个模型都实现了与隐藏层相似的性能，但添加 bigrams 可以显着提高精度。在测试时，Tagspace 需要计算所有类别的分数，这使得它相对较慢，而当类别数量很多（此处超过300K）时，我们的快速推理会显著提高速度。总体而言，获得质量更好的模型的速度要快一个数量级。 测试阶段的加速甚至更加显着（600倍加速）。表4 显示了一些定性的例子。  

## 4 Discussion and conclusion  

在这项工作中，我们提出了一种简单的文本分类基线方法。与来自 word2vec 的无监督训练的单词向量不同，我们的单词特征可以被平均在一起以形成好的句子表示。在几个任务中，fastText 获得的性能与最近提出的深度学习方法相当，而且速度更快。尽管深层神经网络在理论上比浅层模型具有更高的表征能力，但是如何简单的文本分类问题（例如情感分析）来评估它们是不错的。我们将发布我们的代码，以便研究团体可以轻松构建我们的工作。

# Massive Exploration of Neural Machine Translation Architectures  

## Abstract  

过去几年神经机器翻译（NMT）已经显示出显著的进步，现在正在将生产系统部署到最终用户。当前体系结构的一个主要缺点是它们的训练成本很高，通常需要几天到几周的 GPU 时间才能收敛。这使得穷举的超参数搜索（通常与其他神经网络架构一起完成）非常昂贵。在这项工作中，我们首先对 NMT 体系结构超参数进行大规模分析。我们报告了数百次实验运行的实验结果和方差数字，对应于标准 WMT 英语至德语翻译任务超过 250,000 个GPU小时。我们的实验为构建和扩展 NMT 体系结构提供了新颖的见解和实用建议。作为这项贡献的一部分，我们发布了一个开放源代码的 NMT 框架，使研究人员能够轻松实验新技术并重现最先进的结果。  

## 1 Introduction  

神经机器翻译（NMT）（Kalchbrenner 和 Blunsom，2013; Sutskever 等，2014; Cho等，2014）是自动翻译的端到端方法。NMT 已经取得了令人瞩目的成果（Jean et al，2015; Luong et al，2015b; Sennrich et al，2016a; Wu et al，2016），超越了基于短语的系统，同时解决了诸如需要手工特征等缺点。NMT 最流行的方法是基于由两个递归神经网络（RNN）组成的编码器-解码器体系结构以及将目标与源标记对齐的注意机制（Bahdanau 等，2015; Luong等，2015a）。  

目前的 NMT 体系结构的一个缺点是训练它们所需的计算量。对数百万个示例的实际数据集进行训练通常需要数十个 GPU，并且收敛时间大约为几天到几周。虽然扫描大型超参数空间在计算机视觉中很常见（Huang et al，2016b），但这种探索对于 NMT 模型来说过于昂贵，将研究人员限制在完善的架构和超参数选择上。此外，还没有关于架构超参数如何影响 NMT 系统性能的大规模研究。因此，目前仍不清楚为什么这些模型的表现与他们一样好，以及我们如何改进。  

在这项工作中，我们首次对神经机器翻译系统的架构超参数进行综合分析。我们总共使用超过 250,000 小时的 GPU 时间，探索 NMT 架构的常见变体，并深入了解哪些架构选择最重要。我们报告所有实验的 BLEU 分数，困惑度，模型大小和收敛时间，包括在每个实验的几次运行中计算的方差数。另外，我们向公众发布了一个用于运行实验的新软件框架。  

总之，这项工作的主要贡献如下：  

- 我们提供有关神经机器翻译模型优化的有用见解，并为未来的研究提供有希望的方向。例如，我们发现深度编码器比解码器更难以优化，dense 残差连接比常规残差连接产生更好的性能，LSTM 优于 GRU，并且良好调整的 beam search 对于获得最先进的结果至关重要。通过为选择基线架构提供实用建议，我们帮助研究人员避免在未经验证的模型变化上浪费时间。  

- 我们还确定了 BLEU 等指标受随机初始化和轻微超参数变化影响的程度，帮助研究人员将统计上显著的结果与随机噪声区分开来。  

- 最后，我们发布基于 TensorFlow 的开放源代码软件包，专门设计用于实现可重现的序列-序列模型。所有的实验都是使用这个框架进行的，我们希望通过向公众公布加速未来的研究。我们还发布了本文中重现实验所需的所有配置文件和处理脚本。  

## 2 Background and Preliminaries  

![Aaron Swartz](https://github.com/liyibo/cv_notebooks/blob/master/markdown_pics/NLP/6/1.jpg?raw=true)

### 2.1 Neural Machine Translation  

我们的模型基于具有注意力机制的 encoder-decoder 架构（Bahdanau等，2015; Luong等，2015a），如图1 所示。编码器函数 $f_{enc}$ 将源 tokens 序列 $x = (x_1,...,x_m)$ 作为输入，并产生一系列状态 $h = (h_1,...,h_m)$。 在我们的基础模型中，$f_{enc}$ 是一个双向的 RNN，而状态 $h_i$ 对应于由后向和前向 RNN 产生的状态的拼接，$h_i = [\overrightarrow h_i; \overleftarrow h_i]$。解码器 $f_{dec}$ 是基于 h 预测目标序列 $y = (y_1,...,y_k)$ 概率的 RNN。基于解码器 RNN 中的循环状态 $s_i$，前一个词 $y_{<i}$ 和上下文向量 $c_i$ 来预测每个目标 token $y_i \in 1,..,V$ 的概率。上下文向量 $c_i$ 也被称为注意力向量，并被计算为源状态的加权平均值。  

$c_i = \sum_j a_{i,j}h_j \tag{1}$
$a_{i,j} = \frac {\hat a_{i,j}}{\sum_j \hat a_{i,j}} \tag{2}$
$\hat a_{i,j} = att(s_i,h_i) \tag{3}$

这里，$att(s_i,h_i)$ 是计算编码器状态 $h_j$ 与解码器状态 $s_i$ 之间的非标准化对准分数的注意力函数。在我们的基本模型中，我们使用形式为 $att(s_i,h_i) = \left\langle W_h h_j,W_s s_i \right\rangle$ 的函数，其中矩阵 W 用于将 source and target states 转换为相同大小的表示。  

解码器在固定大小 V 的词汇表上输出分布：  

$P(y_i|y_1,...,y_{i-1},x) = softmax(W[s_i；c_i] + b)$

通过使用随机梯度下降来最小化目标词的负对数似然性，整个模型被端对端地训练。  

## 3 Experimental Setup  

### 3.1 Datasets and Preprocessing  

我们通过结合 Europarl v7，新闻评论 v10 和 Common Crawl 语料库组成 4.5M 句子，在对 WMT’15 English→German 任务进行所有实验。我们使用newstest2013 作为我们的验证集，newstest2014 和 newstest2015 作为我们的测试集。为了检验通用性，我们还进行了一些关于英语→法语翻译的实验，并且我们发现该表现与英语→德语的表现高度相关，但是在较大的英语→法语数据集上训练模型需要更长的时间。考虑到形态更丰富的德语的翻译也被认为是一项更具挑战性的任务，我们认为使用英语→德语翻译任务来完成超参数扫描是合理的。  

我们使用 Moses2 中的脚本 tokenize 并清理所有数据集，并使用 32,000 次合并操作使用字节对编码（BPE）学习共享子字单位，最终词汇量大约为 37k。我们发现数据预处理会对最终数据产生很大影响，并且由于我们希望实现可重复性，因此我们将数据预处理脚本与 NMT 框架一起发布给公众。有关数据预处理参数的更多详细信息，请参阅读者的代码版本。  

### 3.2 Training Setup and Software  

以下所有实验均使用我们自己的基于 TensorFlow 的软件框架运行（Abadi et al，2016）。我们特意构建了这个框架，以实现神经机器翻译架构的可重现的最先进的实现。作为我们贡献的一部分，我们正在发布重现我们结果所需的框架和所有配置文件。训练在 Nvidia Tesla K40m 和 Tesla K80 GPU 上进行，每个实验分布在 8 个并行 worker 和 6 个参数服务器上。我们使用 128 的批量大小并使用 beam search 进行解码，beam 宽度为 10，并且（Wu等人，2016）中描述了长度归一化惩罚值 0.6。BLEU 分数是使用 Moses3 中的 multi-bleu.perl 脚本对分词数据进行计算的。每个实验最多运行 2.5M steps，并使用不同的初始化复制 4 次。我们每 30 分钟保存一次模型检查点，并根据验证集 BLEU 得分选择最佳检查点。我们报告每个实验的均值和标准差以及最高分数（按照交叉验证）。  

### 3.3 Baseline Model  

根据对以前文献的回顾，我们选择了一个我们知道能够相当好地执行的基线模型。我们的目标是保持基线模型简单和标准，而不是推进start of the art。该模型（2.1中描述）由一个 2 层双向编码器（每个方向 1 层）和一个带乘法（Luong et al。，2015a）注意机制的 2 层解码器组成。我们对编码器和解码器使用 512-unit GRU（Cho等人，2014），并在每个单元的输入处应用 0.2 的 Dropout。我们使用 Adam 优化器进行训练，固定学习率为 0.0001 而不衰减。嵌入维度设置为 512。可以在补充材料中找到所有模型超参数的更详细描述。  

在以下每个实验中，除了正在研究的一个超参数外，基线模型的超参数保持不变。我们希望这可以让我们隔离各种超参数变化的影响。我们认识到此过程不考虑超参数之间的相互作用，并且当我们认为可能发生此类交互时，我们会执行其他实验。  

## 4 Experiments and Discussion  

为了简洁起见，我们只报告平均 BLEU，标准偏差，最佳 BLEU 以及下表中的模型大小。Log 困惑度，tokens/sec 和收敛时间可以在补充材料表中找到。  

### 4.1 Embedding Dimensionality  

![Aaron Swartz](https://github.com/liyibo/cv_notebooks/blob/master/markdown_pics/NLP/6/2.jpg?raw=true)

由于词汇量很大，嵌入层可以占模型参数的很大一部分。历史上，研究人员使用 620 维（Bahdanau等，2015）或 1024 维（Luong等，2015a）嵌入。我们预计更大的嵌入会导致更好的 BLEU 分数，或至少更低的困惑，但我们发现并非总是如此。虽然表1 显示 2048 维嵌入产生了总体最好的结果，但他们只改善了一点。即使小 128 维嵌入也表现出色，但收敛速度几乎快两倍。我们发现对小嵌入和大嵌入的梯度更新没有显著差异，并且嵌入矩阵的梯度更新的范数在整个训练中保持大致恒定，而不管大小如何。我们也没有观察到大嵌入的过拟合，并且训练的 log 困惑度在整个实验中大致相同，这表明该模型没有有效地使用额外的参数，并且可能需要更好的优化技术。或者，可能会出现这种情况：具有大嵌入的模型只需要超过250万步就可以收敛到最佳解决方案。  

### 4.2 RNN Cell Variant  

![Aaron Swartz](https://github.com/liyibo/cv_notebooks/blob/master/markdown_pics/NLP/6/3.jpg?raw=true)

在 NMT 架构中常常使用 LSTM（Hochreiter和Schmidhuber，1997）和 GRU（Cho等，2014）单元。虽然有一些研究（Greff等，2016）研究了几千例小序列任务中的 cell 变异，但我们并不了解大规模 NMT 环境中的这类研究。  

门控 cells 如 GRU 和 LSTM 的动机是梯度消失问题。使用 vanilla RNN cell，深层网络不能有效地通过多层和时间步骤传播信息和梯度。然而，在基于注意力的模型中，我们认为解码器应该能够几乎完全基于当前输入和注意上下文做出决定，并且我们假设解码器中的 gating 机制不是绝对必要的。这个假设得到以下事实的支持：我们总是将解码器状态初始化为零而不是传递编码器状态，这意味着解码器状态不包含有关编码源的信息。我们通过在解码器中使用 vanilla RNN cell来测试我们的假设。对于 LSTM 和 GRU 变体，我们替换编码器和解码器中的单元。我们使用没有 peephole connections 的 LSTM 细胞，并将 LSTM 和 GRU 细胞的遗忘偏差初始化为1。  

在我们的实验中，LSTM cell 一贯优于 GRU cell。由于我们体系结构中的计算瓶颈是 softmax 操作，因此我们没有观察到 LSTM 和 GRU 单元之间的训练速度差别很大。有些令我们吃惊的是，我们发现 vanilla 解码器无法像门控变体一样学习得很好。这表明解码器确实在多个时间步骤中以其自身状态传递信息，而不是仅仅依赖于注意机制和当前输入（其包括先前的关注上下文）。也可能是门控机制需要掩盖输入中不相关的部分。  

### 4.3 Encoder and Decoder Depth  

![Aaron Swartz](https://github.com/liyibo/cv_notebooks/blob/master/markdown_pics/NLP/6/4.jpg?raw=true)

我们通常期望更深层次的网络能够比浅层网络更好地融合（He et al，2016）。尽管一些工作（Luong等，2015b; Zhou等，2016; Luong和Manning，2016; Wu等，2016）已经使用深度网络取得了最先进的成果，但其他人使用浅层网络（Jean等，2015 ; Chung等人，2016; Sennrich等人，2016b）已经取得了类似的结果。因此，不清楚深度的重要性以及浅层网络是否能够产生与深层网络相竞争的结果。在这里，我们探讨了高达 8 层的编码器和解码器深度的影响。对于双向编码器，我们分别在两个方向上堆叠 RNN。例如，Enc-8 模型对应于一个前向和一个后向 4 层 RNN。对于更深层次的网络，我们还试验了两种残差连接方式（He et al，2016），以鼓励梯度流。在标准变体中，如等式（4）所示，我们在连续层之间插入残差连接。如果 $h_t^{(l)}(x_t^{(l)},h_{t-1}^{(l)})$ 是时间步 t 的第 l 层的 RNN 输出，则：  

$x_t^{(l+1)} = h_t^{(l)}(x_t^{(l)},h_{t-1}^{(l)}) + x_t^{(l)}\tag{4}$

其中 $x_t^{(0)}$ 是输入tokens 的词嵌入。 、我们还探索了一种类似于（Huang et al。、，2016a）在图像识别中使用的残差连接的（“ResD”）dense 变体。在这个变体中，我们添加从每个图层到所有其他图层的 skip 连接：  

$x_t^{(l+1)} = h_t^{(l)}(x_t^{(l)},h_{t-1}^{(l)}) + 、sum_{j=0}^l x_t^{(j)}\tag{4}$

我们的实现不同于（Huang et al，2016a），因为我们使用一个加法而不是一个连接操作来保持状态大小不变。  

表3 示出了编码器和解码器不同深度以及具有和不具有残差连接的结果。我们没有找到明确的证据表明编码器深度超过两层是必要的，但发现具有残留连接的深层模型在训练期间更有可能发散。正如大标准偏差所暗示的，最好的深度残差模型取得了良好的结果，但四个运行中只有一个收敛。  

在解码器方面，深层模型的表现略胜一筹，我们发现没有残差连接，我们不可能训练 8 层或更多层的解码器。在深度解码器实验中，密集的残差连接始终优于常规残差连接，并且在步数方面收敛得更快，如图2 所示。我们期望深度模型表现更好（Zhou et al，2016; Szegedy et al， 2015），我们相信我们的实验证明需要更强大的技术来优化深度序列模型。例如，我们可能需要一个更优化的 SGD 优化器或某种形式的批量标准化，以强健地训练具有残差连接的深度网络。  

![Aaron Swartz](https://github.com/liyibo/cv_notebooks/blob/master/markdown_pics/NLP/6/5.jpg?raw=true)

### 4.4 Unidirectional vs. Bidirectional Encoder  

在文献中，我们使用双向编码器（Bahdanau等，2015），单向编码器（Luong等，2015a）以及两者的混合（Wu等，2016）。双向编码器能够创建考虑过去和未来输入的表示，而单向编码器只能考虑过去的输入。单向编码器的好处是它们的计算可以很容易地在 GPU 上并行化，从而使它们的运行速度比双向同步更快。我们没有意识到任何研究探索双向性的必要性。在这组实验中，我们探索了具有和不具有反向源输入的不同深度的单向编码器，因为这是一种常用的技巧，允许编码器为较早的字创建更丰富的表示。鉴于解码器端的错误可以轻易级联，早期字词的正确性会产生不成比例的影响。  

![Aaron Swartz](https://github.com/liyibo/cv_notebooks/blob/master/markdown_pics/NLP/6/6.jpg?raw=true)

表4 显示双向编码器通常优于单向编码器，但不会大幅度提高。带有反向信号源的编码器的性能始终优于非反向信号，但不会击败较浅的双向编码器。  

### 4.5 Attention Mechanism  

![Aaron Swartz](https://github.com/liyibo/cv_notebooks/blob/master/markdown_pics/NLP/6/7.jpg?raw=true)

两种最常用的注意机制是 additive 变体（Bahdanau等，2015），下面的等式（6），以及计算上较便宜的乘法变体（Luong等，2015a），下面的等式（7）。给定注意 key $h_j$（编码器状态）和注意力查询 $s_i$（解码器状态），每对的注意力分数计算如下：  

$score(h_j,s_i) = \left\langle v,tanh(W_1h_j + W_2s_i) \right\rangle \tag{6}$
$score(h_j,s_i) = \left\langle W_1h_j , W_2s_i \right\rangle \tag{7}$

我们将 W1hj 和 W2si 的维度称为“注意维度”，并通过更改图层大小将其从 128 改为 1024。我们还尝试通过使用最后一个编码器状态（无状态）初始化解码器状态或将最后一个解码器状态连接到每个解码器输入（无输入）来使用无关注机制。结果如表5 所示。  

我们发现参数化加性注意机制略微但一致地优于乘法注意机制，注意维度几乎没有影响。  

尽管我们确实期望基于关注的模型在没有关注机制的情况下显著优于那些模型，但我们对“Non-Input”模型表现差强人意，因为他们在每个时间步可以访问编码器信息。此外，我们发现基于关注的模型在整个训练过程中对解码器状态显示出更大的梯度更新。这表明，注意机制更像是一种“加权跳过连接”，它优化了梯度流程，而不像文献中通常所述的允许编码器访问源状态的“存储器”。我们认为在这个方向上的进一步研究对于阐明关注机制的作用是否有必要，以及它是否可能纯粹是一种简化优化的工具。  

### 4.6 Beam Search Strategies  

Beam Search 是一种常用的技术，通过树搜索来最大化一些评分函数 $s(y,x)$找到目标序列。在最简单的情况下，被最大化的得分是给定源的目标序列的对数概率。最近，诸如覆盖惩罚（Tu等人，2016）和长度归一化（Wu等人，2016）的扩展已经被用来改进解码结果。还观察到（Tu et al，2017），即使长度不足，非常大的 Beam Search 尺寸也比较小的 Beam Search 尺寸差。因此，选择正确的 Beam Search 宽度对获得最佳结果至关重要。  

![Aaron Swartz](https://github.com/liyibo/cv_notebooks/blob/master/markdown_pics/NLP/6/8.jpg?raw=true)

表6 显示了改变 Beam Search 宽度和增加长度归一化惩罚的影响。beam width 1 对应于贪婪搜索。我们发现调整好的 Beam Search 对于获得好的结果是至关重要的，并且它会导致不止一个 BLEU 点的一致增益。类似于（Tu et al，2017），我们发现非常大的 Beam 会产生更差的结果，并且存在最佳光束宽度的“最佳点”。我们认为对 Beam Search 中超参数的鲁棒性的进一步研究对 NMT 的进展至关重要。我们还尝试了一个覆盖率惩罚，但没有发现超过足够长的惩罚额外收益。  

### 4.7 Final System Comparison  

![Aaron Swartz](https://github.com/liyibo/cv_notebooks/blob/master/markdown_pics/NLP/6/9.jpg?raw=true)
![Aaron Swartz](https://github.com/liyibo/cv_notebooks/blob/master/markdown_pics/NLP/6/10.jpg?raw=true)

最后，我们将在 newstest2013 验证集上选择的所有实验（具有512维加性关注的基本模型）的最佳表现模型与表8 中的文献中的历史结果进行比较。虽然不关注此项工作，但通过将我们所有的见解与表7 中描述的单一模型相结合，能够实现进一步的改进。  

尽管我们不提供架构创新，但我们确实通过仔细的超参数调整和良好的初始化来证明，可以在标准 WMT 基准测试中实现最先进的性能。我们的模型仅受到（Wu et al，2016）的表现的影响，该模型更复杂。  

## 5 Open Source Release  

我们凭经验证明了超参数值和不同初始化的微小变化是如何影响结果的，以及如何调整 beam search 等看似微不足道的因素至关重要。为了实现可重复的研究，我们认为研究人员必须开始建立共同的框架和数据处理流程。考虑到这一目标，我们专门构建了一个模块化软件框架，使研究人员能够以最少的代码更改探索新型架构，并以可重现的方式定义实验参数。虽然我们的初始实验是在机器翻译中，但我们的框架可以很容易地适应 Summarization，会话建模或图像到文本中的问题。像 OpenNMT 这样的系统（Klein et al，2017）有相似的目标，但尚未达到最先进的结果（见表8），缺乏我们认为是关键特征的因素，如分布式训练支持。我们希望通过开源我们的实验工具包，使该领域在未来取得更快的进展。  

我们的所有代码均可在 https://github.com/google/seq2seq/ 免费获取。  

## 6 Conclusion  

我们进行了我们认为是神经机器翻译的第一个大规模建筑架构分析，剔除了实现最先进结果的关键因素。我们展示了一些令人惊讶的见解，其中包括 beam search 调整与大多数架构变化同样重要的事实，并且使用当前优化技术，深层模型并不总是胜过浅层。在这里，我们总结了我们的实际发现：  

- 2048尺寸的大型嵌入获得了最佳效果，但只有小幅度的提升。即使 128 维的小嵌入似乎也有足够的能力来捕获大部分必要的语义信息。
- LSTM Cells 始终优于 GRU Cells。
- 具有 2 至 4 层的双向编码器性能最佳。更深层次的编码器对于训练来说显然更加不稳定，但是如果它们能够很好地优化，则显示出潜力。
- 4 层深解码器略胜于更浅的解码器。残差连接对训练 8 层解码器是必要的，密集的残差连接提供了额外的稳健性。
- 参数化的 additive 注意力产生了总体最佳结果。
- 调整良好的附加长度惩罚的波束搜索至关重要。5 到 10 的光束宽度以及 1.0 的长度惩罚似乎工作良好。  

我们强调了几个重要的研究问题，包括有效利用嵌入参数（4.1），注意机制作为加权跳过连接（4.5）而不是内存单元的作用，需要更好的深度循环网络优化方法（4.3），并且需要对超参数变化具有鲁棒性的更好的波束搜索（4.6）。  

此外，我们还向公众发布了一个开放源代码的 NMT 框架，专门用于探索架构创新并生成可重复的实验，以及用于我们所有实验的配置文件。

# A Neural Attention Model for Abstractive Sentence Summarization  

## Abstract  

基于文本提取的摘要本质上是有限的，但生成式 abstractive 方法已被证明具有挑战性。在这项工作中，我们提出了一种完全数据驱动的 abstractive 句子摘要方法。我们的方法利用基于局部注意力的模型，该模型生成以输入句子为条件的摘要的每个单词。虽然该模型结构简单，但可以轻松地进行端到端的训练，并可以扩展到大量的训练数据。与几个强大的基线相比，该模型显示了 DUC-2004 共享任务的显著性能提升。  

## 1 Introduction  

![Aaron Swartz](https://github.com/liyibo/cv_notebooks/blob/master/markdown_pics/NLP/7/1.jpg?raw=true)

Summarization 是自然语言理解的重要挑战。目的是产生输入文本的浓缩表示，捕获原始的核心含义。最成功的摘要系统利用 extractive 方法，将文本的部分裁剪和拼接在一起以产生浓缩版本。相反，abstractive 摘要试图产生自下而上的摘要，其中的 aspects 可能不会作为原始的一部分出现。  

我们专注于句子级摘要的任务。虽然在这项任务上的大量工作已经研究了基于删除的句子压缩技术（Knight和Marcu（2002），以及许多其他人），但 human summarizers 的研究表明，在压缩时应用其他各种操作是常见的，例如释义，泛化和重新排序（Jing，2002）。过去的工作已经使用语言启发的约束（Dorr等，2003; Zajic等，2004）或输入文本的句法转换（Cohn和Lapata，2008; Woodsend等，2010）来模拟这种 abstractive 概括问题。这些方法在第6 节中有更详细的描述。  

我们改为探索一种完全数据驱动的方法来生成 abstractive 摘要。受近期神经机器翻译成功的启发，我们将神经语言模型与上下文输入编码器相结合。我们的编码器模仿了 Bahdanau 等人的基于注意力的编码器。因为它在输入文本上学习潜在的软对齐以帮助形成摘要（如图1所示）。至关重要的是编码器和生成模型在句子摘要任务上共同训练。该模型在第 3 节中有详细描述。我们的模型还包括 beam-search 解码器以及提取元素模型的附加特征；这些方面将在第 4 节和第 5 节中讨论。  

![Aaron Swartz](https://github.com/liyibo/cv_notebooks/blob/master/markdown_pics/NLP/7/2.jpg?raw=true)

这种 Summarization 方法（我们称之为 Attention-Based Summarization（ABS））比可比较的 abstractive 概括方法包含更少的语言结构，但可以轻松地扩展以训练大量数据。由于我们的系统不对生成的摘要的词汇表做出任何假设，因此可以直接在任何文档-摘要对上进行训练。这使我们能够在Gigaword 的文章对语料库中训练标题生成的摘要模型，它由约 400 万篇文章组成。图 2 给出了生成示例，我们将在第 7 节中讨论此任务的详细信息。  

为了测试这种方法的有效性，我们对多个 abstractive 和提取基线进行了广泛的比较，包括传统的基于语法的系统，整数线性程序约束系统，信息检索样式方法，以及基于统计短语的机器翻译。第 8 节描述了这些实验的结果。我们的方法优于在相同的大规模数据集上训练的机器翻译系统，并且在 DUC-2004 竞赛中比最高评分系统产生了很大的改进。  

## 2 Background  

我们首先定义句子摘要任务。给定一个输入句子，目标是产生一个简明的摘要。让输入由来自大小为 |V| 的固定词汇表 V 中的 M 个字 $x_1,...,x_M$ 的序列组成。我们将每个单词表示为指示符向量 $x_i \in \{0,1\}^V$，其中 $i\in \{1,...,M\}$，句子作为一系列指示符，X 作为可能输入的集合。此外，定义符号 $x_{[i,j,k]}$ 以指示元素 $i,j,k$ 的子序列。  

摘要器将 $x$ 作为输入并输出长度为 $N<M$ 的缩短句子。我们假设摘要中的单词也来自相同的词汇表 V，并且输出是序列 $y_1,...,y_N$。请注意，与相关任务（如机器翻译）相比，我们假设输出长度 N 是固定的，并且系统在生成之前知道摘要的长度。  

接下来考虑生成摘要的问题。将集合 $Y \subset ({0,1}^V,...,{0,1}^V)$ 定义为长度为 N 的所有可能句子，即对于所有 i 和 y∈Y，yi 是指示符。我们说如果一个系统试图从这个集合 Y中找到最优序列，那么它就是 abstractive 的，  

${arg \ max}_{y \in Y} s(x,y) \tag{1}$

将此与 fully extractive sentence summary 对比，后者从输入中传递单词：  

${arg \ max}_{m \in \{1,...,M\}^N} s(x,x_{[m_1,...,m_N]}) \tag{2}$

或者相关的句子压缩问题，集中于从输入中删除单词：  

${arg \ max}_{m \in \{1,...,M\}^N,m_{i-1} < m_i} s(x,x_{[m_1,...,m_N]}) \tag{3}$

虽然 abstractive 摘要带来了更困难的生成挑战，但缺乏硬约束使系统更自由地生成，并允许它适应更广泛的训练数据。  

在这项工作中，我们专注于因子评分函数 s，它考虑了以前单词的固定窗口：  

$s(x,y) \approx \sum_{i=0}^{N-1} g(y_{i+1},x,y_c) \tag{4}$

我们为大小为 C 的窗口定义 $y_c = y_{[i-C+1,...,i]}$。  

特别考虑给定输入的摘要的条件对数概率，$s(x,y) = \log p(y|x,\theta)$。我们可以这样写：  

$\log p(y|x,\theta) \approx \sum_{i=0}^{N-1} \log p(y_{i+1}|x,y_c,\theta) \tag{4}$

我们在上下文的长度上做马尔可夫假设为大小 C 并假设 $i<1$，$y_i$ 是一个特殊的起始符号 $\langle S \rangle$。  

考虑到这个评分函数，我们主要关注的是对局部条件分布进行建模 $p(y_{i+1}|x,y_c,\theta)$。下一节定义了这个分布的参数化，在第 4 节中，我们回到了因式模型的生成问题，在第 5 节中我们介绍了一个修改的因式评分函数。  

## 3 Model  

![Aaron Swartz](https://github.com/liyibo/cv_notebooks/blob/master/markdown_pics/NLP/7/3.jpg?raw=true)

感兴趣分布 $p(y_{i+1}|x,y_c,\theta)$ 是基于输入句子 x 的条件语言模型。过去关于摘要和压缩的工作使用噪声通道方法来分割和独立估计语言模型和条件摘要模型（Banko等，2000; Knight和Marcu，2002; Daume III和'Marcu，2002）。在这里，我们遵循神经机器翻译的工作，并直接将原始分布参数化为神经网络。该网络包含神经概率语言模型和充当条件概括模型的编码器。  

### 3.1 Neural Language Model   

我们参数化的核心是用于估计下一个单词的上下文概率的语言模型。语言模型改编自标准的前馈神经网络语言模型（NNLM），特别是 Bengio 等人描述的NNLM类。（2003年）。 完整的模型是：  

$p(y_{i+1}|x,y_c,\theta) ∝ \exp (Vh + Wenc(x,y_c)) , \ \tilde y_c = [Ey_{i-C+1},...,Ey_i], \ h=tanh(U \tilde y_c)$

参数是 $\theta =(E,U,V,W)$ 其中 $E \in R^{D×V}$ 是一个字嵌入矩阵，$U \in R^{(CD) \times H}，$V \in R^{V \times H}，$W \in R^{V \times H}$ 是权重矩阵，D 是单词嵌入的大小，h 是大小为 H 的隐藏层。黑盒函数 enc 是上下文编码器项，它返回表示输入和当前上下文的大小为 H 的向量；我们考虑随后描述的几种可能的变体。图 3a 给出了解码器架构的示意图。  

### 3.2 Encoders  

请注意，如果没有编码器术语，则表示标准语言模型。通过结合使用和共同训练这两个元素，我们至关重要的是可以将输入文本合并到 generation 中。我们将讨论编码器的几个可能的实例化。  

**Bag-of-Words编码器** 我们最基本的模型只使用嵌入到 H 大小的输入句子的词袋，而忽略原始顺序的属性或相邻单词之间的关系。我们把这个模型写成：  

$enc_1 (x,y_c) = p^T \tilde x, \ p = [1/M,...,1/M], \ \tilde x = [Fx_1,...,Fx_M]$

在输入侧嵌入矩阵 $F \in R^{H×V}$ 是编码器的唯一新参数且 $p \in [0,1]^M$ 是输入字上的均匀分布的情况下。  

对于摘要，该模型可以捕获单词的相对重要性，以区分内容单词和停用单词或装饰。模型也可以学习组合单词；虽然它在表示连续短语方面具有内在的局限性。  

**卷积编码器** 为了解决词袋建模的一些问题，我们还考虑使用深度卷积编码器来输入句子。这种体系结构通过允许单词之间的局部交互来改进单词袋模型，同时在编码输入时也不需要上下文 yc。  

我们利用标准时延神经网络（TDNN）架构，在时间卷积层和最大池化层之间交替。  

![Aaron Swartz](https://github.com/liyibo/cv_notebooks/blob/master/markdown_pics/NLP/7/4.jpg?raw=true)

其中 F 是字嵌入矩阵，$QL×H×2Q + 1$由每层的一组滤波器组成。式。Eq.7 是时间（1D）卷积层，Eq.6 由 2 元素时间最大汇集层和逐点非线性以及最终输出组成。Eq.5 is a max over time。每层的 $\tilde x$ 是 $\overline x$ 大小的一半。为简单起见，我们假设卷积在边界处填充，并且 M 大于 $2^L$，因此尺寸定义明确。  

**基于注意力的编码器** 虽然卷积编码器具有比 bag-ofwords 有更丰富的容量，但仍然需要为整个输入句子产生单个表示。机器翻译中的类似问题启发了 Bahdanau 等人，改为使用基于注意的上下文编码器，其基于生成上下文构造表示。在这里我们注意到，如果我们利用这个上下文，我们实际上可以使用一个类似于词袋的相当简单的模型：  

![Aaron Swartz](https://github.com/liyibo/cv_notebooks/blob/master/markdown_pics/NLP/7/5.jpg?raw=true)

其中 $G \in R^{D×V}$ 是上下文的嵌入，$P \in R^{H×(CD)}$是一个新的权重，是上下文嵌入和输入嵌入之间的矩阵参数映射，Q是平滑窗口。完整模型如图 3b 所示。  

非正式地，我们可以将此模型视为简单地用输入和摘要之间的学习软对齐 P 替换词袋中的均匀分布。图1显示了生成摘要时此分布的示例。然后，在构造表示时，使用软对齐来对输入的平滑版本进行加权。该模型可被视为基于注意力的神经机器翻译模型的精简版本。  

### 3.3 Training  

缺乏生成约束使得可以在任意输入-输出对上训练模型。一旦我们定义了局部条件模型 $p(y_{i+1}|x,y_c,\theta)$，我们就可以估计参数，以最小化一组摘要的负对数似然。将此训练集定义为由 J 输入-摘要对 $(x^1,y_1),...,(x^J,y^J)$。负对数似然方便地将摘要中的每个 token factors into a term：  

$NLL(\theta) = -\sum_{j=1}^J \log p(y^{(j)}|x^{(j)};\theta) = -\sum_{j=1}^J \sum_{i=1}^{N-1} \log p(y_{i+1}^{(j)}|x^{(j)},y_c;\theta)$

我们通过使用小批量随机梯度下降来最小化 NLL。细节将在第 7 节中进一步描述。  

## 4 Generating Summaries  

我们现在回到生成摘要的问题。回想一下 Eq.4 我们的目标是找到，  

$y^* = {arg \ max}_{y \in Y} \sum_{i=0}^{N-1} g(y_{i+1},x,y_c)$

与基于短语的机器翻译不同，其中推理是 NP 困难的，理论上它在计算 $y^*$ 时实际上是易处理的。由于没有明确的硬对齐约束，因此可以应用维特比解码并且需要 $O(NV^C)$ 时间来找到精确解。在实践中，尽管 V 足够大以使其变得困难。另一种方法是用严格贪婪或确定性的解码器来近似 arg max。  

精确解码和贪婪解码之间的折衷是使用 beam-search 解码器（算法1），其保持完整词汇表 V，同时将其自身限制在摘要的每个位置处的 K 个潜在假设。 这是神经 MT 模型的标准方法（Bahdanau等，2014; Sutskever等，2014; Luong等，2015）。此处显示了波束搜索算法，针对前馈模型进行了修改：  

![Aaron Swartz](https://github.com/liyibo/cv_notebooks/blob/master/markdown_pics/NLP/7/6.jpg?raw=true)

与维特比一样，这种波束搜索算法比基于短语的 MT 的波束搜索简单得多。因为没有明确的约束条件，每个源字都可以使用一次，不需要保持位设置，我们可以简单地从左到右生成字。波束搜索算法需要 $O(KNV)$ 时间。然而，从计算的角度来看，每轮波束搜索由针对 K 个假设中的每一个的计算 $p(yi|x,yc)$。这些可以作为小批量计算，实际上大大降低了 K 的因子。  

## 5 Extension: Extractive Tuning  

虽然我们将看到基于注意力的模型在生成摘要时是有效的，但它确实错过了 human-generated 的参考文献中看到的一个重要方面。特别地，abstractive 模型在必要时不具有找到提取词匹配的能力，例如从输入中转移看不见的专有名词短语。在神经翻译模型中也观察到类似的问题，特别是在翻译稀有词汇方面（Luong等，2015）。  

为了解决这个问题，我们尝试调整一小组附加特征，这些特征可以权衡系统的 abstractive/extractive 趋势。我们通过修改我们的评分函数来直接使用对数线性模型估计汇总概率，这是机器翻译的标准。  

## 6 Related Work  

abstractive 句子摘要传统上与标题生成任务相关。我们的工作类似于 Banko 等人的早期工作，使用标题-文章对的语料库为这项任务开发了一种统计机器翻译启发式方法。我们通过以下方式扩展了这种方法：（1）使用神经摘要模型而不是基于计数的噪声信道模型，（2）在更大规模上训练模型（25K与400万篇文章相比），（3）并允许完全 abstractive 的解码。  

该任务在 DUC-2003 和 DUC-2004 竞赛中被标准化（Over等，2007）。TOPIARY 系统（Zajic等，2004）在此任务中表现最佳，将在下一节中详细介绍。我们将感兴趣的读者指向 DUC 网页（ http://duc.nist.gov/ ），以获取在此共享任务中输入的系统的完整列表。  

最近，Cohn 和 Lapata（2008）给出了一种允许更多任意变换的压缩方法。他们从对齐的，解析的文本中提取树转换规则，并使用最大边缘学习算法学习转换的权重。伍德森等人提出了一种准同步语法方法，利用无上下文解析和依赖解析来产生清晰的摘要。这两种方法与我们的不同之处在于它们直接使用输入/输出句子的语法。后一个系统在我们的结果中是 W＆L；我们试图在此数据集上训练前系统 T3，但无法对其进行大规模训练。  

除了 Banko 等人，已经有一些工作直接使用统计机器翻译进行 abstractive Summarization 。Wubben 等直接利用 MOSES 作为文本简化的方法。  

最近，Filippova 和 Altun（2013）开发了一种严格的提取系统，该系统在一个相对较大的语料库（250K句子）的四边形对上进行训练。因为它们的重点是提取压缩，所以通过一系列启发式转换句子，使得单词处于单调对齐中。我们的系统不需要这个对齐步骤，而是直接使用文本。  

**Neural MT** 这项工作与神经网络语言模型（NNLM）的最新研究和神经机器翻译工作密切相关。我们模型的核心是基于 Bengio 等人的 NNLM。  

最近，有几篇关于机器翻译模型的论文（Kalchbrenner和Blunsom，2013; Cho等，2014; Sutskever等，2014）。其中我们的模型与 Bahdanau 等人的基于注意力的模型最密切相关。与前馈模型相反，这些模型中的大多数使用递归神经网络（RNN）进行生成。我们希望在未来的工作中加入 RNNLM。  

## 7 Experimental Setup  

我们在标题生成任务上尝试基于注意力的句子摘要模型。在本节中，我们将描述用于此任务的语料库，我们与之比较的基线方法以及我们方法的实现细节。  

### 7.1 Data Set  

标准句子摘要评估集与 DUC-2003 和 DUC-2004 共享任务相关联（Over等，2007）。这项任务的数据包括来自纽约时报和美联社有线服务的 500 篇新闻文章，每篇文章都配有 4 个不同的人工参考摘要（实际上不是标题），上限为 75 字节。该数据集仅供评估，但类似大小的 DUC-2003 数据集可用于该任务。基于完整文章的文本（尽管我们仅使用第一句），期望大约 14 个单词的摘要。完整的数据集可通过 http://duc.nist.gov/data.html 索取。  

对于此共享任务，使用面向召回的 ROUGE 指标的几种变体输入和评估系统（Lin，2004）。为了使仅召回评估不受长度影响，所有系统的输出在 75 个字符后被截止，并且对于较短的摘要没有给予奖励。与插入各种 n-gram 匹配的 BLEU 不同，ROUGE 有多种版本可用于不同的匹配长度。DUC 评估使用 ROUGE-1（unigrams），ROUGE-2（bigrams）和ROUGE-L（最常见的子串），我们报告了所有这些。  

除了标准的 DUC-2014 评估之外，我们还报告了使用 Gigaword 的随机保留子集对单个参考标题生成的评估。此评估更接近于模型训练的任务，它允许我们使用更大的评估集，我们将在代码版本中包含该评估集。对于此评估，我们调整系统以生成平均标题长度的输出。  

对于两个任务的训练数据，我们使用带注释的 Gigaword 数据集（Graff等，2003; Napoles等，2012），其由标准 Gigaword 组成，使用 Stanford CoreNLP 工具预处理（Manning等，2014）。我们的模型仅使用注释进行 tokenization 和句子分离，尽管一些基线也使用 parsing 和 tagging。在过去二十年中，Gigaword 包含来自各种国内和国际新闻服务的约 950 万条新闻文章。  

对于我们的训练集，我们将每篇文章的标题与其第一句配对以创建输入对。虽然该模型理论上可以在任何一对上进行训练，但 Gigaword 包含许多虚假的标题-文章对。因此，我们根据以下启发式过滤器修剪训练：（1）是否没有共同的非停止词？（2）标题是否包含署名或其他无关的编辑标记？（3）标题是否带有问号或冒号？ 应用这些过滤器后，训练集大约包含 J = 400 万个标题-文章对。我们应用最小的预处理步骤，使用 PTB 标记化，lower-casing，用＃替换所有数字字符，并用 UNK 替换少于 5 次的单词类型。我们还删除了 DUC 评估期间的所有文章。  

完整的输入训练词汇包括 1.19 亿个 word tokens 和 110K 个独特单词类型，平均句子大小为 31.3 个单词。标题词汇包括 3100 万个 tokens 和 69K 个单词类型，平均标题长度为 8.3 个单词（请注意，这比 DUC 摘要要短得多）。在标题和输入之间平均有 4.6 种重叠的单词类型；虽然在输入的前 75 个字符中只有 2.6。  

### 7.2 Baselines  

由于句子摘要问题的方法多种多样，我们报告了一系列标题生成基线。  

从 DUC-2004 任务中我们包括 PREFIX 基线，它只返回输入的前 75 个字符作为标题。我们还报告了这个共享任务 TOPIARY 的 winning 系统（Zajic等，2004）。TOPIARY 使用输入的语言动机转换（Dorr等人，2003）和无监督主题检测（UTD）算法合并压缩系统，该算法将完整文章中的关键短语附加到压缩输出上。  

DUC 任务还包括由 8 个人类摘要者执行的一组手动摘要，每个摘要 Summarization 一半的测试数据句子（每个句子产生4个参考）。我们将平均注释间协议分数报告为 REFERENCE。作为参考，最佳人类评估者得分为 31.7 ROUGE-1。  

我们还包括几个可以访问与我们的系统相同的训练数据的基线。第一个是句子压缩基线 COMPRESS（Clarke和Lapata，2008）。该模型使用原始句子的句法结构以及在标题数据上训练的语言模型来产生压缩输出。语法和语言模型与一组语言约束相结合，并且使用 ILP 求解器执行解码。  

为了控制记忆训练中的标题，我们实施了信息检索基线 IR。该基线对训练集进行索引，并给出与输入具有最高 BM-25 匹配的文章的标题。  

最后，我们使用基于短语的 Gigaword 训练的统计机器翻译系统来生成摘要，MOSES +（Koehn et al，2007）。为了改善此任务的基线，我们使用“删除”规则扩充短语表，将每个文章单词映射到 $\epsilon$，包括这些规则的附加删除功能，并允许无限的失真限制。我们还使用 MERT 显式调整模型，以针对 75 字节的上限 ROUGE 分数，而不是标准的基于BLEU的调整。不幸的是，剩下的一个问题是修改转换解码器以产生固定长度输出是非常重要的，因此我们调整系统以产生大致预期的长度。  

### 7.3 Implementation  

对于训练，我们使用小批量随机梯度下降来最小化负对数似然。我们使用 0.05 的学习率，并且如果验证对数似然在一个时期没有改善，则将学习率分成一半。使用尺寸为 64 的小批量进行训练。小批量按输入长度分组。在每个 epoch 之后，我们重新规范嵌入表（Hinton等，2012）。 基于验证集，我们将超参数设置为 D = 200，H = 400，C = 5，L = 3 和 Q = 2。  

我们的实现使用 Torch numerical 框架（ http://torch.ch/ ），并将与数据管道一起公开提供。至关重要的是，训练是在 GPU 上进行的，并且难以处理或需要近似值。处理 1000 个小批量，D = 200，H = 400 需要 160 秒。通过数据在 15 个时期之后达到最佳验证准确度，这需要大约 4 天的训练。  

另外，如第 5 节所述，我们在使用 DUC-2003 数据进行训练后应用 MERT 调整步骤。对于这一步，我们使用 Z-MERT（Zaidan，2009）。我们将主模型称为ABS，将调整后的模型称为 ABS+。  

## 8 Results  

![Aaron Swartz](https://github.com/liyibo/cv_notebooks/blob/master/markdown_pics/NLP/7/7.jpg?raw=true)

我们的主要结果如表1 所示。我们使用 DUC-2004 评估数据集（500个句子，4个参考，75个字节）和所有系统以及随机保持的 Gigaword 测试集（2000个句子，1个参考）进行实验。我们首先注意到基线 COMPRESS 和 IR 在两个数据集上的表现相对较差，这表明仅仅有文章信息或语言模型信息都不足以完成任务。PREFIX 基线实际上在 ROUGE-1 上表现出色，这在前面观察到的文章和摘要之间的重叠是有意义的。  

ABS 和 MOSES+ 都比 TOPIARY 表现更好，特别是在 DUC 中的 ROUGE-2 和 ROUGE-L 上。完整模型 ABS+ 在这些任务上得分最高，并且基于所有指标上的TOPIARY 的默认 ROUGE 置信水平以及 DUC 的 ROUGE-1 上的 MOSES+ 以及 Gigaword 的 ROUGE-1 和 ROUGE-L 明显更好。请注意，额外的提取功能会使系统偏向于保留更多输入字，这对于基础指标非常有用。  

接下来我们考虑消融模型和算法结构。表2 显示了具有各种编码器的模型的实验。对于这些实验，我们将系统的困惑视为验证数据的语言模型，验证数据控制推理和调整的变量。没有编码器的 NNLM 语言模型比标准的 n-gram 语言模型有所增加。甚至包括词袋编码器也将误差数减少到 50 以下。卷积编码器和基于注意力的编码器都进一步减少了困惑，注意力提供了低于 30 的值。  

![Aaron Swartz](https://github.com/liyibo/cv_notebooks/blob/master/markdown_pics/NLP/7/8.jpg?raw=true)

我们还考虑了主要 summary 模型上的模型和解码消融，如表3 所示。这些实验与 BoW 编码模型进行比较，比较波束搜索和贪婪解码，以及限制系统完全提取。在这些特征中，最大的影响是使用更强大的编码器（注意力与BoW），以及使用波束搜索生成摘要。系统的 abstractive 性质有所帮助，但对于 ROUGE 来说，即使使用纯粹的提取生成也是有效的。  

![Aaron Swartz](https://github.com/liyibo/cv_notebooks/blob/master/markdown_pics/NLP/7/9.jpg?raw=true)

最后，我们考虑图4 中所示的示例摘要。尽管基线得分有所改善，但该模型远非人类在此任务上的表现。通常，模型擅长从输入中挑选关键词，例如名称和地点。但是，两个模型都会以语法错误的方式重新排序单词，例如在 Sentence 7 中，两个模型都有错误的主题。ABS 经常使用更有趣的重写措辞，例如在第 4 new nz pm after election，但这也可能导致 attachment 错误，如 Sentence11 中的 russian oil giant chevron。  

## 9 Conclusion  

基于神经机器翻译的最新发展，我们提出了一种基于神经注意的 abstractive 摘要模型。我们将此概率模型与生成算法相结合，生成算法可生成准确的abstractive 摘要。作为下一步，我们希望以数据驱动的方式进一步改进摘要的语法，并扩展此系统以生成段落级摘要。两者在有效排列和 generation 一致性方面都提出了额外的挑战。

# Abstractive Text Summarization Using Sequence-to-Sequence RNNs and Beyond  

## Abstract  

在这项工作中，我们使用 Attentional EncoderDecoder Recurrent Neural Networks 对抽象文本摘要进行建模，并表明它们在两个不同的语料库中实现了最先进的性能。我们提出了几种新颖的模型来解决摘要中的关键问题，这些问题没有被基本架构充分建模，例如建模关键词，捕获句子结构的层次结构，以及找到训练时罕见或看不见的单词。我们的工作表明，我们提出的许多模型有助于进一步提高性能。我们还提出了一个由多句子摘要组成的新数据集，并为进一步研究建立了性能基准。  

## 1 Introduction  

抽象文本摘要是生成标题或简短摘要的任务，该摘要由捕获文章或段落显著想法的几个句子组成。我们使用形容词 'abstractive' 来表示一个摘要，它不仅仅是从源中提取的一些现有段落或句子的选择，而是对文档主要内容的压缩释义，可能使用源文档中未见的词汇。  

此任务也可以自然地转换为将源文档中的单词的输入序列映射到称为摘要的单词的目标序列。在最近的过程中，基于深度学习的模型将输入序列映射到另一个输出序列，称为序列到序列模型，在机器翻译（Bahdanau et al，2014），语音识别等许多问题上都取得了成功。Bahdanau等，2015）和视频字幕（Venugopalan等，2015）。在序列到序列模型的框架中，我们的任务的一个非常相关的模型是 Bahdanau 等人提出的注意力递归神经网络（RNN）编码器解码器模型，它在机器翻译（MT）中产生了最先进的性能，这也是一种自然的语言任务。  

尽管有相似之处，但抽象摘要与 MT 是一个非常不同的问题。与MT不同，目标（摘要）通常非常短，并且在很大程度上不依赖于摘要中源（文档）的长度。 另外，摘要中的关键挑战是以有损方式最优地压缩原始文档，使得保留原始文档中的关键概念，而在 MT 中，预期转换是无损的。在翻译中，源和目标之间存在几乎一对一的词级对齐的强烈概念，但在摘要中，它不那么明显。  

我们在这项工作中做出了以下主要贡献：（i）我们将最初为机器翻译开发的现成的注意力编码器-解码器 RNN 应用于摘要，并表明它在两个不同的英语语料库已经超越了最先进的系统 。（ii）由于基于机器翻译的模型未充分解决的摘要中的具体问题，我们提出新颖的模型并表明它们提供了性能的额外改进。（iii）我们提出了一个新的数据集，用于将文档抽象概括为多个句子并建立基准。  

本文的其余部分安排如下。在第2节中，我们描述了我们旨在解决的抽象概括中的每个特定问题，并提出了一个解决它的新模型。第3节将我们的模型与关于抽象文本摘要主题的密切相关的工作进行了语境化。我们在第4节中介绍了我们在三个不同数据集上的实验结果。我们还在第5节中对模型输出进行了定性分析，然后在第6节中对我们未来的发展方向进行了评论。  

## 2 Models  

在本节中，我们首先描述作为基线的基本编码器编码器 RNN，然后提出几种用于摘要的新颖模型，每种模型都解决了基线中的特定弱点。  

### 2.1 Encoder-Decoder RNN with Attention and Large Vocabulary Trick  

我们的基线模型对应于 Bahdanau 等人使用的神经机器翻译模型。编码器由双向 GRU-RNN（Chung等，2014）组成，而解码器由具有与编码器相同的隐藏状态大小的单向 GRU-RNN，source-hidden states 的注意力机制和目标词汇表上的 soft-max 层以生成单词。我们将读者引用到原始论文中，以便对该模型进行详细处理。除了基本模型之外，我们还适应了摘要问题，Jean 等人描述的大词汇“技巧”（LVT）。在我们的方法中，每个 mini-batch 的解码器词汇仅限于该批次的源文档中的单词。此外，添加目标词典中最常用的单词，直到词汇量达到固定大小。该技术的目的是减小解码器的 soft-max 层的大小，这是主要的计算瓶颈。此外，该技术还通过将建模工作仅集中在给定示例所必需的单词上来加速收敛。这种技术特别适合于摘要，因为摘要中的大部分单词在任何情况下都来自源文档。  

### 2.2 Capturing Keywords using Feature-rich Encoder  

![Aaron Swartz](https://github.com/liyibo/cv_notebooks/blob/master/markdown_pics/NLP/8/1.jpg?raw=true)

在摘要中，关键挑战之一是确定文档中的关键概念和关键实体，故事围绕这些概念和关键实体展开。为了实现这一目标，我们可能需要超越基于单词嵌入的输入文档表示，并捕获其他语言特征，如词性标记，命名实体标记以及 TF 和 IDF 统计数据。因此，我们为每个标签类型的词汇创建了额外的基于查找的嵌入矩阵，类似于单词的嵌入。对于 TF 和 IDF 等连续特征，我们通过将它们离散化为固定数量的区间将它们转换为分类值，并使用 one-hot 表示来指示它们落入的区间数。这允许我们将它们映射到嵌入矩阵，就像任何其他标记类型一样。最后，对于源文档中的每个单词，我们只需从其所有相关标签中查找其嵌入并将它们连接成一个长向量，如图1 所示。在目标端，我们继续只使用基于单词嵌入的表示。  

### 2.3 Modeling Rare/Unseen Words using Switching Generator-Pointer  

![Aaron Swartz](https://github.com/liyibo/cv_notebooks/blob/master/markdown_pics/NLP/8/2.jpg?raw=true)

通常在摘要中，对于摘要而言，测试文档中的关键字或命名实体对于训练数据实际上可能是看不见的或罕见的。由于解码器的词汇在训练时是固定的，所以它不能输出这些看不见的词。相反，处理这些词典外（OOV）单词的最常用方法是输出 'UNK' 标记作为占位符。但是，这不会产生清晰的摘要。在摘要中，处理此类 OOV 字的直观方式是简单地指向它们在源文档中的位置。我们使用我们新颖的 switching decoder/pointer 架构对此信号进行建模，如图2 所示。在此模型中，解码器配备了一个“开关”，用于决定在每个时间步使用发生器还是指针。如果开关打开，则解码器以正常方式从其目标词汇表中产生一个单词。但是，如果关闭开关，则解码器生成指向源中的一个字位置的指针。然后将指针位置处的单词复制到摘要中。基于每个时间步长的整个可用上下文，将开关建模为线性层上的 S 形激活函数，如下所示。  

$P(s_i = 1) = \sigma(V^s \cdot (W_h^s h_i + W_e^s E[o_{i-1}] + W_c^s c_i + b^s))$

其中 $P(s_i = 1)$ 是在解码器的第 $i$ 个时间步骤开关导通的概率，$h_i$ 是隐藏状态，$E[o_{i-1}]$ 是前一时间步的输出的嵌入矢量 ，$c_i$ 是注意加权的上下文向量，$W_h^s, W_e^s, W_c^s, b^s$ 和 $v^s$ 是切换参数。我们使用文档中单词位置的注意力分布作为从中采样指针的分布。  

在训练时，只要目标词汇表中不存在摘要词，我们就为模型提供显式指针信息。当摘要中的 OOV 字出现在多个文档位置时，我们打破平局，支持其首次出现。  

指针机制在处理稀有单词时可能更加健壮，因为它使用编码器的罕见单词的隐藏状态表示来决定文档中指向哪个单词。由于隐藏状态取决于单词的整个上下文，因此模型能够准确地指向看不见的单词，尽管它们不出现在目标词汇表中。  

### 2.4 Capturing Hierarchical Document Structure with Hierarchical Attention  

![Aaron Swartz](https://github.com/liyibo/cv_notebooks/blob/master/markdown_pics/NLP/8/3.jpg?raw=true)

在源文档很长的数据集中，除了识别文档中的关键字之外，识别可以从中绘制摘要的关键句子也很重要。该模型旨在使用两个双向 RNN 捕获这两个重要级别的概念，一个在单词级别，另一个在句子级别。注意机制同时在两个层面上运作。通过相应的句子级注意进一步重新加权词级注意，并重新规范化。  

此外，我们还将附加位置嵌入连接到句子级 RNN 的隐藏状态，以模拟文档中句子的位置重要性。因此，该架构共同模拟关键句子以及这些句子中的关键词。该模型的图形表示如图3 所示。  

## 3 Related Work  

绝大多数过去的摘要工作都是提取式的，其中包括确定源文件中的关键句子或段落，并将其作为摘要再现（Neto等，2002; Erkan和Radev，2004; Wong等，2008a; Filippova 和Altun，2013; Colmenares等，2015; Litvak和Last，2008; K. Riedhammer和Hakkani-Tur，2010; Ricardo Ribeiro，2013）。  

另一方面，人类倾向于用自己的话来解释原始故事。因此，人类摘要本质上是抽象的，很少从文档中复制原始句子。抽象概括的任务已经使用 DUC-2003 和 DUC-2004 竞赛进行了标准化。这些任务的数据包括来自各种主题的新闻摘要，每个摘要由人类生成。DUC-2004 任务中表现最佳的系统称为 TOPIARY（Zajic等，2004），它结合了语言驱动的压缩技术和无监督的主题检测算法，该算法将从文章中提取的关键字附加到压缩输出上。抽象概括任务中的一些其他值得注意的工作包括使用传统的基于短语表的机器翻译方法（Banko等，2000），使用加权树转换规则进行压缩（Cohn和Lapata，2008）和准同步语法。  

随着深度学习作为许多 NLP 任务的可行替代方案的出现（Collobert等，2011），研究人员已开始将此框架视为一种有吸引力的，完全数据驱动的抽象摘要替代方案。Rush 等人使用卷积模型对源进行编码，并使用上下文敏感的注意力前馈神经网络生成摘要，在 Gigaword 和 DUC 数据集上生成最先进的结果。 在这项工作的扩展中，Chopra 等人对编码器使用了类似的卷积模型，但用 RNN 替换了解码器，从而进一步提高了两个数据集的性能。  

在另一篇与我们的工作密切相关的论文中，Hu 等人引入了一个用于中文短文摘要的大型数据集。他们使用编码器-解码器 RNN 在中文数据集上显示了有希望的结果，但没有报告英语语料库的实验。  

在另一项最近的工作中，Cheng 和 Lapata（2016）使用基于 RNN 的编码器-解码器来提取文档的摘要。这个模型与我们的模型不具有直接可比性，因为它们的框架是抽取式的，而（Rush等，2015），（Hu等，2015）和（Chopra等，2016）是抽象式的。  

我们的工作从与（Hu et al，2015）相同的框架开始，我们在源和目标上使用 RNN，但我们超越了标准体系结构并提出了解决摘要中关键问题的新模型。 我们还注意到这项工作是 Nallapati 等人的扩展版本。除了与该工作相比进行更广泛的实验外，我们还提出了一个新的文档摘要数据集，我们也在其上建立了基准数字。  

下面，我们分析我们提出的模型与相关摘要工作的异同。  

**Feature-rich encoder**（第2.2节）：诸如 POS 标签，命名实体以及 TF 和 IDF 信息等语言特征被用于许多提取方法的摘要（Wong et al，2008b），但据我们所知，它们在深度学习方法的抽象式摘要中是新颖的。  

**Switching generator-pointer model**（第2.3节）：该模型将提取和抽象方法结合到单个端到端框架中进行汇总。Rush 等人也使用了提取和抽象方法的组合，但它们的提取模型是一个单独的对数线性分类器，具有手工制作的特征。指针网络（Vinyals et al。，2015）也被用于机器翻译中罕见词语的问题（Luong et al，2015），但在我们的模型中新增的开关允许它在何时忠实于原始来源（例如，对于命名实体和OOV）以及何时允许其具有创造性之间取得平衡。我们相信这样一个过程可以说是模仿人类如何产生摘要。有关此模型的更详细处理以及多项任务的实验，请参阅本工作的一些作者发表的并行工作（Gulcehre等，2016）。  

**Hierarchical attention model**（第2.4节）：以前提出的分层编码器-解码器模型仅在句子层面使用注意力（Li et al，2015）。我们的方法的新颖之处在于对句子和单词级别的注意力的联合建模，其中单词级别的注意力被句子级别的注意力进一步影响，从而捕获重要句子和这些句子中的重要单词的概念。 位置嵌入与句子级隐藏状态的串联也是新的。  

## 4 Experiments and Results  

### 4.1 Gigaword Corpus  

在这一系列实验中，我们使用 Rush 等人所述的带注释的 Gigaword 语料库。我们使用本工作作者提供的脚本来预处理数据，从而产生了大约 3.8M 的训练样例。该脚本还生成大约 400K 验证和测试示例，但我们创建了一个随机抽样的子集，每个示例包含 2000 个示例，每个示例用于验证和测试目的，我们在其上报告我们的性能。此外，我们还获得了 Rush 等人使用的精确测试样本。将我们的模型与他们的模型进行精确比较。我们还对脚本进行了少量修改，不仅提取了 tokenized 的单词，还提取了系统生成的词性和命名实体标记。  

**训练** 对于我们下面讨论的所有模型，我们使用 200 维 word2vec 向量（Mikolov等，2013）在同一语料库上训练来初始化模型嵌入，但我们允许它们在训练期间更新。在我们的所有实验中，编码器和解码器的隐藏状态维数固定为 400。当我们仅使用文档的第一句作为源时，如 Rush 等人所做的那样。编码器词汇量大小为 119,505，解码器词汇量为 68,885。我们使用 Adadelta（Zeiler，2012）进行训练，初始学习率为 0.001。我们使用 50 的批量大小并在每个时期随机 shuffled 训练数据，同时根据它们的长度对每 10 批进行分类以加速训练。我们没有使用任何 dropout 或正则化，但应用了渐变裁剪。我们使用基于验证集的早期停止，并使用验证集上的最佳模型来报告所有测试性能数字。对于我们所有的模型，我们采用大词汇技巧，我们将解码器词汇量限制为 2,0005，因为它将每个时期的训练时间减少了近三倍，并帮助这个和所有后续模型融合在一起仅基于完整词汇量的模型所需的时期的 50％-75％。  

**解码** 在解码时，我们使用大小为 5 的波束搜索来生成摘要，并将摘要的大小限制为最多 30 个单词，因为这是我们在采样验证集中注意到的最大大小。我们发现，我们所有模型（7.8到8.3）的平均系统摘要长度与验证集（约8.7个单词）的 ground truth 非常接近，没有任何特定的调整。  

**计算成本** 我们在一台 Tesla K40 GPU 上训练了所有模型。大多数模型平均每个时期花费大约 10 个小时，除了分层注意模型，每个时期需要 12 个小时。所有模型通常使用我们基于验证成本的早期停止标准在 15 个时期内收敛。因此，收敛的训练时间在 6-8 天之间变化，具体取决于模型号。在测试时生成摘要的速度相当快，单个 GPU 上的吞吐量大约为每秒 20 个摘要，批量大小为 1。  

**评估指标** 与（Nallapati等，2016）和（Chopra等，2016）类似，我们使用 Rouge 的全长 F1 变体来评估我们的系统。虽然有限的长度召回是大多数先前工作的首选度量，但其缺点之一是选择长度限制，从语料库到语料库不等，使得研究人员难以比较性能。另一方面，全长召回不会限制长度限制，但不公平地支持更长的摘要。全长 F1 解决了这个问题，因为它可以惩罚更长的摘要，而不会强加特定的长度限制。  

此外，我们还报告了源中发生的系统摘要中 tokens 的百分比。我们在下面的 Gigaword 语料库中描述了我们的所有实验和结果。  

words-lvt2k-1sent：这是具有大量词汇技巧的基线注意力编码器-解码器模型。该模型仅在源文档的第一句中进行训练，如 Rush 等人所述。  

words-lvt2k-2sent：这个模型与上面的模型完全相同，只是它是根据源的前两个句子进行训练的。在这个语料库中，在源代码中添加附加句子似乎有助于提高性能，如表1 所示。我们还尝试添加更多句子，但性能下降，这可能是因为此语料库中的后面句子与摘要没关系。  

![Aaron Swartz](https://github.com/liyibo/cv_notebooks/blob/master/markdown_pics/NLP/8/4.jpg?raw=true)

words-lvt2k-2sent-hieratt：由于我们使用了源文档中的两个句子，我们训练了第 2.4 节中提出的层次关注模型。如表1 所示，通过自动学习前两个句子的相对重要性，该模型与其更平坦的对应物相比提高了性能。  

feats-lvt2k-2sent：在这里，我们仍然训练前两个句子，但是我们利用带注释的千兆字库语料库中的词性和命名实体标签以及 TF，IDF 值来增强输入嵌入。如2.2节所述的源端。总的来说，我们的嵌入向量从最初的 100 增长到 155，并且与表1中所示的对应单词 lvt2k-2sent 相比产生了增量增益，证明了基于语法的特征在该任务中的实用性。  

feats-lvt2k-2sent-ptr：这是 2.3 节中描述的交换发生器/指针模型，但此外，我们还在文档方面使用特征丰富的嵌入，如上面的模型。我们的实验表明，新模型能够在我们的测试集上获得所有三种 Rouge 变体的最佳性能，如表1 所示。  

**与 state-of-the-art 技术的比较** 我们比较了 Rush 等人创建的样本中我们的模型 words-lvt2k- 1sent 和最先进模型的表现，如表1 底部所示。我们还训练了另一个系统，我们称之为 words-lvt5k-1sent，它具有更大的 LVT 词汇量 5k，但也有更大的源和目标词汇量400K和200K。  

我们之前没有评估我们最好的验证模型的原因是这个测试集只包含源文档中的 1 个句子，并且不包括我们最好的模型中需要的 NLP 注释。该表显示，尽管如此，我们的模型优于 Rush 等人的 ABS+ 模型。此外，我们的模型表现出更好的抽象能力，如 src 所示。此外，我们较大的模型 words-lvt5k-1sent 优于最先进的模型（Chopra等，2016），对 Rouge-1 有统计学上的显着改善。  

我们相信，我们用于对源进行建模的双向 RNN 捕获了比 Rush 等人使用的 bag-of-embeddings 表示更丰富的每个单词的上下文信息。在他们的卷积式注意编码器中，这可能解释了我们的优越性能。此外，重要信息的显式建模，如多源语句，词级语言特征，使用切换机制指向需要时的源词，以及层次关注，解决摘要中的特定问题。  

### 4.2 DUC Corpus  

![Aaron Swartz](https://github.com/liyibo/cv_notebooks/blob/master/markdown_pics/NLP/8/5.jpg?raw=true)

DUC 语料库分为两部分：2003 语料库，包括 624 个文档，摘要对和 2004 年语料库，由 500 对组成。由于这些语料库太小而无法训练大型神经网络，Rush 等人（2015）在 Gigaword 语料库上训练他们的模型，但是将其与具有手工特征的附加对数线性提取摘要模型相结合，该模型在 DUC 2003 语料库上训练。他们称原始神经注意模型为 ABS 模型，组合模型为 ABS+。Chopra 等也报告了他们的 RASElman 模型在该语料库上的表现，并且是当前最先进的，因为它优于所有先前公布的基线，包括基于非神经网络的提取和抽象系统。  

在我们的工作中，我们只是按原样运行在 Gigaword 语料库上训练的模型，而无需在 DUC 验证集上进行调整。我们对解码器进行的唯一更改是禁止模型发出摘要结束标记，并强制它为每个摘要发出正好 30 个单词，因为此语料库的官方评估基于有限长度的 Rouge 召回。在这个语料库中，由于我们只有一个来自源的句子而没有 NLP 注释，我们只运行了模型 words-lvt2k-1sent 和 words-lvt5k-1sent。  

该模型在测试上的性能与 ABS 和 ABS+ 模型，RASElman 以及表2 中 DUC-2004 的最佳表现系统 TOPIARY 进行了比较。我们注意到我们的最佳模型在 Rouge 的三个变体中的两个上优于 RAS-Elman，同时在 Rouge-1 上具有竞争力。  

### 4.3 CNN/Daily Mail Corpus  

![Aaron Swartz](https://github.com/liyibo/cv_notebooks/blob/master/markdown_pics/NLP/8/6.jpg?raw=true)

现有的抽象文本摘要语料库（包括Gigaword和DUC）在每个摘要中只包含一个句子。在本节中，我们提出了一个包含多项摘要的新语料库。为了产生这种语料库，我们修改了用于基于通道的问答的任务的现有语料库（Hermann等，2015）。在这项工作中，作者使用来自 CNN 和 Daily Mail 网站中的新故事的人类生成的抽象摘要作为问题（隐藏了其中一个实体），并将故事作为系统预期应答填充的相应段落空白的问题。作者发布了从这些网站抓取，提取和生成一对段落和问题的脚本。通过对脚本的简单修改，我们以原始顺序恢复每个故事的所有摘要项目符号，以获得多句子摘要，其中每个项目符号被视为句子。总的来说，这个语料库有 286,817 个训练对，13,368 个验证对和 11,487 个测试对，由他们的脚本定义。训练集中的源文档平均有 766 个单词，平均 29.74 个句子，而摘要由 53 个单词和 3.72 个句子组成。这个数据集的独特特征，如长文档和有序的多句子摘要提出了有趣的挑战，我们希望将吸引未来的研究人员在其上构建和测试新的模型。    

![Aaron Swartz](https://github.com/liyibo/cv_notebooks/blob/master/markdown_pics/NLP/8/7.jpg?raw=true)

## 5 Qualitative Analysis  

![Aaron Swartz](https://github.com/liyibo/cv_notebooks/blob/master/markdown_pics/NLP/8/8.jpg?raw=true)
![Aaron Swartz](https://github.com/liyibo/cv_notebooks/blob/master/markdown_pics/NLP/8/9.jpg?raw=true)

表5 显示了来自 feats-lvt2k-2sent（我们表现最佳的模型之一）的验证集上的一些高质量和低质量输出。即使模型与目标摘要不同，其摘要也往往非常有意义且相关，这种现象未被词汇/短语匹配评估指标（如Rouge）捕获。另一方面，该模型有时会“误解”文本的语义，并生成一个带有滑动解释的摘要，如表中质量不佳的示例所示。显然，捕捉复杂句子的“含义”仍然是这些模型的弱点。  

![Aaron Swartz](https://github.com/liyibo/cv_notebooks/blob/master/markdown_pics/NLP/8/10.jpg?raw=true)

我们的下一个示例输出（如图4所示）显示了 Gigaword 语料库上的切换生成器/指针模型的样本输出。从这些例子中可以明显看出，模型学会非常准确地使用指针，不仅对于命名实体，而且对于多词短语。尽管其准确性，但整体模型的性能改进并不显著。我们相信这种模型的影响可能在其他环境中更为明显，尾部分布较重的稀有单词。我们打算将来用这个模型进行更多的实验。  

在美国有线电视新闻网/每日邮报数据中，虽然我们的模型能够生成高质量的多句子摘要，但我们注意到相同的句子或短语经常在摘要中重复出现。我们相信包含内部注意力的模型，如 Cheng 等可以通过鼓励模型“记住”过去已经产生的单词来解决这个问题。  

## 6 Conclusion  

在这项工作中，我们将注意力 encoder-decoder 应用于抽象摘要的任务，结果非常有希望，在两个不同的数据集上显著优于最先进的结果。我们提出的每个新模型都解决了抽象摘要中的特定问题，从而进一步提高了性能。我们还提出了一个新的多数据摘要数据集，并在其上建立基准数字。作为我们未来工作的一部分，我们计划将精力集中在这些数据上，并为包含多个句子的摘要构建更强大的模型。

# QANet: Combining Local Convolution with Global Self-Attention for Reading Comprehension  
 

## ABSTRACT  

当前的端到端机器读取和问答（Q＆A）模型主要基于具有注意力的递归神经网络（RNN）。尽管取得了成功，但由于 RNN 的序列特性性，这些模型对于训练和推理通常都很慢。我们提出了一种名为 QANet 的新 Q＆A 架构，它不需要循环网络：它的编码器完全由卷积和 self-attention 组成，其中卷积模拟局部交互和自我关注模拟全局交互。在 SQuAD 数据集上，我们的模型在训练中速度提高了 3 到 13 倍，推理速度提高了 4 到 9 倍，同时实现了与循环模型相当的精度。加速增益使我们能够使用更多数据训练模型。因此，我们将模型与来自神经机器翻译模型的反向翻译生成的数据相结合。在 SQUAD 数据集上，我们使用增强数据训练的单一模型在测试集上获得了 84.6 F1 得分，这明显优于最佳公布的 F1 得分 81.8。  

## 1 INTRODUCTION  

人们对机器阅读理解和自动问答的任务越来越感兴趣。在过去几年中，端到端模型取得了重大进展，在许多具有挑战性的数据集上显示了有希望的结果。最成功的模型通常采用两个关键因素：（1）处理顺序输入的循环模型，以及（2）应对长期交互的注意组件。这两种成分的成功组合是 Seo 等人的双向注意力流动（BiDAF）模型。在 SQuAD 数据集上取得了很好的成果。这些模型的一个缺点是，由于它们的循环特性，它们通常对于训练和推理都很慢，特别是对于长文本。昂贵的训练不仅导致实验的周转时间长，而且限制了研究人员快速迭代，但也阻止了模型用于更大的数据集。同时，慢速推断会阻止机器理解系统部署在实时应用程序中。  

在本文中，为了使机器理解速度快，我们建议消除这些模型的重复性。相反，我们专门使用卷积和自我注意作为编码器的构建块，它们分别对查询和上下文进行编码。然后我们通过标准注意来学习语境和问题之间的相互作用（Xiong等，2016; Seo等，2016; Bahdanau等，2015）。在最终解码为每个位置作为答案跨度的开始或结束的概率之前，使用我们的 recurrency-free 编码器再次对所得到的表示进行编码。我们将此架构称为 QANet，如图1 所示。  

我们模型设计背后的关键动机如下：卷积捕获文本的局部结构，而自我注意力学习每对词之间的全局交互。附加的上下文查询注意力是为上下文段落中的每个位置构造查询感知上下文向量的标准模块，其在随后的建模层中使用。我们架构的前馈特性显著加速了模型。在我们对 SQuAD 数据集的实验中，我们的模型在训练中快 3 到 13 倍，在推理中快 4 到 9 倍。作为一个简单的比较，我们的模型可以在 3 小时的训练中达到与 BiDAF 模型（Seo等人，2016）相同的准确度（77.0 F1得分），否则应该花费 15 个小时。加速增益还允许我们通过更多迭代训练模型，以获得比竞争模型更好的结果。例如，如果我们允许我们的模型训练 18 小时，它在 dev set 上获得了 82.7 的 F1 分数，这比（Seo等，2016）好得多，并且与最佳公布结果相当。  

由于我们的模型很快，我们可以使用比其他模型更多的数据来训练它。为了进一步改进模型，我们提出了一种补充数据增强技术来增强训练数据。这种技术通过将原始句子从英语翻译成另一种语言然后再翻译成英语来解释这些例子，这不仅增加了训练实例的数量，而且使语法多样化。  

在 SQuAD 数据集上，使用增强数据训练的 QANet 在测试集上获得 84.6 F1 分数，这明显优于 Hu 等人的最佳公布结果 81.8。我们还进行消融测试，以证明我们模型的每个组成部分的有用性。总之，本文的贡献如下：  

- 我们提出了一种有效的阅读理解模型，它完全建立在卷积和自我关注的基础之上。据我们所知，我们是第一个这样做的人。与 RNN 相比，这种组合保持了良好的准确性，同时在训练中实现了高达 13 倍的加速和每次训练迭代达到 9 倍。加速增益使我们的模型成为扩展到更大数据集的最有希望的候选者。 

- 为了改善我们在 SQuAD 上的结果，我们提出了一种新颖的数据增强技术，通过释义来丰富训练数据。它允许模型实现比最先进技术更高的精度。  

## 2 THE MODEL  

在本节中，我们首先制定阅读理解问题，然后描述所提出的模型QANet：它是一个前馈模型，仅包含卷积和自我注意，这是一种经验有效的组合，也是我们工作的新颖贡献。  

### 2.1 PROBLEM FORMULATION  

本文考虑的阅读理解任务定义如下。给定具有 n 个单词的上下文段落 $C = \{c_1,c_2,...,c_n\}$ 和具有 m 个单词的查询语句 $Q = \{q_1,q_2,...,q_m\}$，输出来自原始段落C的跨度 $S = \{c_i,c_{i+1},...,c_{i+j}\}$。在下文中，对于任何 $x \in C,Q$，我们将使用 x 来表示原始单词及其嵌入向量。  

### 2.2 MODEL OVERVIEW  

![Aaron Swartz](https://github.com/liyibo/cv_notebooks/blob/master/markdown_pics/NLP/9/1.jpg?raw=true)

我们模型的高级结构类似于包含五个主要组件的大多数现有模型：嵌入层，嵌入编码器层，上下文查询关注层，模型编码器层和输出层，如图1 所示。这些是大多数（如果不是全部）现有阅读理解模型的标准构建块。然而，我们的方法和其他方法之间的主要区别如下：对于嵌入和模型编码器，我们只使用卷积和自我关注机制，丢弃 RNN，这是大多数现有阅读理解模型使用的。因此，我们的模型更快，因为它可以并行处理输入 tokens。请注意，即使自我关注已经在 Vaswani 等人中广泛使用。卷积和自我关注的结合是新颖的，并且明显优于单独的自我关注，并且在我们的实验中获得 2.7 F1 的收益。卷积的使用还允许我们利用 ConvNets 中的常见正则化方法，例如随机深度（layer dropout）（Huang et al，2016），在我们的实验中增加了 0.2 F1。  

详细地说，我们的模型包括以下五个层次：  

**1.输入嵌入层** 我们采用标准技术通过拼接单词嵌入和字符嵌入来获得每个单词 w 的嵌入。单词嵌入在训练期间是固定的，并且从预训练的 GloVe（Pennington等，2014）$p_1 = 300$ 维单词向量初始化。所有 out-of-vocabulary 词汇被映射到一个 $<UNK>$ token，其嵌入是可训练的随机初始化的向量。字符嵌入如下获得：每个字符表示为维度 $p_2 = 200$ 的可训练向量，意味着每个字可以被视为其每个字符的嵌入向量的拼接。每个单词的长度被截断或填充到 16。我们取该矩阵的每一行的最大值来获得每个单词的固定大小的向量表示。最后，来自该层的给定单词 x 的输出是 $[x_w; x_c] \in R^{p_1+p_2}$ 的拼接，其中 $x_w$ 和 $x_c$ 分别是字嵌入和 x 的字符嵌入的卷积输出。与 Seo 等人一样，我们还采用了两层高速公路网络（Srivastava等，2015）。为简单起见，我们还使用 x 来表示该层的输出。  

**2.嵌入编码器层** 编码器层是以下基本构建块的 stack：[卷积层×＃ + 自注意层 + 前馈层]，如图1 右上角所示。我们使用深度可分离卷积而不是传统的，因为我们观察到它的 memory efficient 并具有更好的泛化性。内核大小为 7，过滤器的数量为 d = 128，块内的 conv 层数为 4。对于自注意层，我们采用了（Vaswani等，2017a）定义的多头注意机制，对于输入中的每个位置，称为查询，基于点积测量的查询和 keys 之间的相似性，计算输入中所有位置或 keys 的加权和。所有层中的 heads 数为 8。这些基本操作中的每一个（conv / self-attention / ffn）都放在一个残差块内，如图1 右下图所示。对于输入 x 和给定的操作 f，输出为 $f(layernorm(x))+x$，意味着从每个块的输入到输出存在完整的标识路径，其中 layernorm 表示在（Ba等人，2016）中提出的层标准化。编码器块的总数是1。注意，对于每个单独的字，该层的输入是维度 p1 + p2 = 500 的向量，其通过一维卷积立即映射到 d = 128。该层的输出也是维度 d = 128。  

**3.Context-Query 注意层** 该模块几乎是所有以前的阅读理解模型的标准，如 Weissenborn 等。我们使用 C 和 Q 来表示上下文和查询的编码。上下文到查询的关注构造如下：我们首先计算每对上下文和查询词之间的相似性，呈现相似性矩阵 $S \in R^{n×m}$。然后，我们通过应用 softmax 函数对 S 的每一行进行归一化，得到矩阵 $\overline S$。然后，将上下文 - 查询关注度计算为 $A = \overline S \cdot Q^T \in R^{n×d}$。这里使用的相似函数是 trilinear 函数（Seo et al，2016）：  

$f(q,c) = W_o[q,c,q \bigodot c]$

其中 $\bigodot$ 是 element-wise 乘法，W0 是可训练的变量。  

大多数高性能模型还使用某种形式的查询到上下文的注意力，例如 BiDaF（Seo等人，2016）和 DCN（Xiong等人，2016）。根据经验，我们发现 DCN 注意力可以比简单地应用上下文到查询注意力更好一点，因此我们采用这种策略。更具体地，我们通过 softmax 函数计算 S 的列标准化矩阵 $\overline S$，并且查询到上下文的关注是 $B = \overline S \cdot \overline{\overline S}^T \cdot C^T$。  

**4.模型编码器层** 与 Seo 等人相似，该层在每个位置的输入是 $[c,a,c \bigodot a,c \bigodot b]$，其中 a 和 b 分别是关注矩阵 A 和 B 中的一行。层参数与嵌入编码器层相同，除了块内的卷积层数是2，块的总数是7。我们在模型编码器的 3 次重复中的每一次之间共享权重。  

**5.输出层** 该层是特定于任务的。SQuAD 中的每个示例都在包含答案的上下文中用 span 标记。我们采用了 Seo 等人的策略，预测上下文中每个位置的概率是答案范围的开始或结束。更具体地，起始位置和结束位置的概率被建模为  

$p^1 = sofrmax(W_1[M_0;M_1]), \ p^2 = softmax(W_2[M_0;M-2])$

其中 W1 和 W2 是两个可训练变量，M0，M1，M2 分别是三个模型编码器的输出，从下到上。span 的得分是其起始位置和结束位置概率的乘积。最后，目标函数被定义为由真实开始和结束索引的预测分布的负对数概率和，在所有训练样本上取平均值：  

$L(\theta) = - \frac{1}{N} \sum_{i}^{N} [\log(p_{y_i^1}^{1}) + \log(p_{y_i^2}^{2})]$

其中 $y_i^1$ 和 $y_i^2$ 分别是示例 i groundtruth 的开始和结束位置，$\theta$ 包含所有可训练变量。所提出的模型可以根据其他理解任务进行定制，例如，通过相应地改变输出层从候选答案中进行选择。  

**推理** 在推理时间，通过最大化 $p^1_s p^2_e$ 来选择预测范围 $(s,e)$。标准动态编程可以在线性时间获得线性结果。  

## 3 DATA AUGMENTATION BY BACKTRANSLATION  

![Aaron Swartz](https://github.com/liyibo/cv_notebooks/blob/master/markdown_pics/NLP/9/2.jpg?raw=true)

由于我们的模型很快，我们可以用更多的数据训练它。因此，我们将我们的模型与简单的数据增强技术相结合，以丰富训练数据。这个想法是使用两个翻译模型，一个从英语到法语（或任何其他语言）的翻译模型和另一个从法语到英语的翻译模型，以获得文本的释义。这种方法有助于自动增加任何基于语言的任务的训练数据量，包括我们感兴趣的阅读理解任务。随着更多数据，我们希望更好地规范我们的模型。增强过程如图2 所示，法语作为关键语言。  

在这项工作中，我们考虑基于注意力的神经机器翻译（NMT）模型(Bahdanau et al; Luong et al)，作为我们数据增强管道的核心模型，他们被 Wu 等人证明具有优异的翻译效果。具体来说，我们利用 Luong 等人提供的公开代码库，它复制了谷歌的 NMT（GNMT）系统。我们在公共 WMT 数据上训练 4 层 GNMT 模型，包括英语-法语（36M句子对）和英语-德语（4.5M句子对）。所有数据都已被标记化并分成子字单元，如 Luong 等人所述。所有模型共享相同的超参数，并使用不同的步数进行训练，英语-法语为2M，英语-德语为340K。我们的英语-法语系统在 newstest2014 上实现了 36.7 BLEU。对于英语-德语，在 newstest2014 上，我们获得 27.6 BLEU。  

我们的 paraphrase 过程如下，据说法语是一种关键语言。首先，我们将输入序列馈送到英语-法语模型的波束解码器中以获得 k 个法语翻译。然后，每个法语翻译通过反向翻译模型的波束解码器，以获得输入序列的总共 $k^2$ 个释义。  

![Aaron Swartz](https://github.com/liyibo/cv_notebooks/blob/master/markdown_pics/NLP/9/3.jpg?raw=true)

**相关工作** 虽然之前已经引入了反向翻译的概念，但它通常用于改进相同的翻译任务(Sennrich et al.(2016))，或内在释义评估等(Wieting et al.(2017); Mallinson et al.(2017))。我们的方法是一种新的反向翻译应用，以丰富下游任务的训练数据，在这种情况下，问答（QA）任务。值得注意的是（Dong et al，2017）使用 paraphrasing 技术来改善 QA；然而，他们只是 paraphrase 了问题而没有像我们在本文中那样关注数据增加方面。  

**处理SQuAD文档和答案** 我们现在讨论 SQuAD 数据集的具体处理过程，这对于获得最佳性能提升至关重要。请记住，SQUAD 的每个训练示例都是一个三元组（d，q，a），其中文档 d 是具有答案 a 的多句段落。在 paraphrasing 时，我们保持问题 q 不变（以避免意外地改变其含义）并生成新的三元组$(d',q',a')$，使得新文档 $d'$ 的新答案为 $a'$。该过程分两步进行：（i）document paraphrasing - 将 d 解释为 $d'$ 和（b）答案提取 - 从 $d$' 中提取一个与 a 紧密匹配的 $a'$。  

对于文档释义步骤，我们首先将段落分成句子并独立地 paraphrase 它们。我们使用 k = 5，因此每个句子有 25 个 paraphrase 选择。通过用随机选择的paraphrase 简单地替换 d 中的每个句子来形成新文档 $d'$。这种简单方法的一个明显问题是原始答案 a 可能不再出现在 $d'$ 中。  

答案提取解决了上述问题。设 s 是包含原始答案 a 的原始句子，$s'$ 是其 paraphrase。我们通过简单的启发式识别新的 paraphrased 答案如下。在 $s'$ 中的每个单词和 a 的开始/结束单词之间计算字符级 2-gram 分数，以找到 $s'$ 中答案可能的开始和结束位置。在所有候选 paraphrased 答案中，选择具有相对于 a 的最高字符 2-gram 分数的答案作为新答案 $a'$。表1 显示了此过程发现的新答案的示例。  

paraphrases 的质量和多样性对数据增加方法至关重要。仍然可以改善该方法的质量和多样性。通过使用更好的翻译模型可以提高质量。例如，我们发现paraphrases 比我们模型的最长序列显著更长，最大训练序列长度往往在中间切断。通过波束搜索解码期间的采样和数据集中的释义问题和答案，可以改善多样性。此外，我们可以将此方法与其他数据增强方法（例如，类型交换方法（Raiman＆Miller，2017））结合使用，以获得更多的释义多样性。  

在我们的实验中，我们观察到所提出的数据增强可以在准确性方面带来较大的改进。我们相信这种技术也适用于其他监督的自然语言处理任务，特别是当训练数据不足时。  

## 4 EXPERIMENTS  

在本节中，我们进行实验研究我们的模型和数据增强技术的性能。我们将主要在 SQuAD 数据集（RajPurkar等人，2016）中对我们的模型进行评估，被认为是 Q＆A 中最有竞争力的数据集之一。我们还对 TrviaQA（Joshi等人，2017）进行了类似的研究，来证明我们的模型的有效性和效率是可以泛化的。

### 4.1 EXPERIMENTS ON SQUAD  

#### 4.1.1 DATASET AND EXPERIMENTAL SETTINGS  

**数据集** 我们考虑斯坦福问答数据集（SQUAD）（Rajpurkar等，2016）来进行机器阅读理解。SQuAD 包含 107.7K 查询-答案对，87.5K 用于训练，10.1K 用于验证，另外 10.1K 用于测试。段落的典型长度约为 250，而问题是 10 个 tokens，尽管有特别长的案例。只有训练和验证数据是公开可用的，而测试数据是隐藏的，必须将代码提交给 Codalab 并与（Rajpurkar等人，2016）的作者一起检索最终的测试分数。在我们的实验中，我们报告了我们最好的单一模型的测试集结果。为了进一步分析，我们只报告验证集上的性能，因为我们不想通过频繁提交来探测看不见的测试集。根据我们的实验和之前的工作观察，如（Seo等，2016; Xiong等，2016; Wang等，2017; Chen等，2017），验证得分与测试得分有很大相关性。  

**数据预处理** 我们使用 NLTK 分词器对数据进行预处理。最大上下文长度设置为 400，任何长于该值的段落都将被丢弃。在训练期间，我们按长度批量处理示例，并使用特殊符号 $<PAD>$ 动态填充短句。最大答案长度被设置为 30。我们使用预训练的 300-d 字矢量（Pennington等人，2014），并且所有的 out-of-vocabulary 词汇都用 $<UNK>$替换，其嵌入在训练期间被更新替换。每个字符嵌入随机初始化为 200-D 向量，也在训练中更新。我们生成了从第 3 节获得的另外两个增强数据集，其中包含 140K 和 240K 示例，分别表示为“数据增强×2”和“数据增强×3”，包括原始数据。  

**训练细节** 我们采用两种类型的标准正则化。首先，我们对所有可训练变量使用 L2 权重衰减，参数 $\lambda = 3×10^{-7}$。我们还在字，字符嵌入和层之间使用了 dropout，其中字和字符 dropout rate 分别为 0.1 和 0.05，并且每两层之间的 dropout rate 为 0.1。我们还在每个嵌入或模型编码器层中采用随机深度方法（layer dropout）（Huang等，2016），其中子层 $l$ 具有生存概率 $p_l = 1-\frac{l}{L}(1-p_L)$其中 L 是最后一个层和 $p_L = 0.9$。  

隐藏层大小和卷积滤波器数均为 128，批量大小为 32，原始数据的训练步长为 150K，“数据增强×2”为 250K，“数据增强×3”为 340K。嵌入和模型编码器中的卷积层数为 4 和 2，内核大小为 7 和 5，编码器的 block numbers 分别为 1 和 7。  

我们使用 ADAM 优化器（Kingma＆Ba，2014），其中 $\beta_1 = 0.8,\beta_2 = 0.999 , \epsilon= 10^{-7}$。我们使用学习率预热方案，在前 1000 个步骤中反向指数从 0.0 增加到 0.001，然后在剩余的训练中保持恒定的学习率。指数移动平均值适用于所有可训练变量，衰减率为 0.9999。  

最后，我们使用 Tensorflow（Abadi等，2016）在 Python 中实现我们的模型，并在 NVIDIA p100 GPU 上进行实验。  

#### 4.1.2 RESULTS  

![Aaron Swartz](https://github.com/liyibo/cv_notebooks/blob/master/markdown_pics/NLP/9/4.jpg?raw=true)

**Accuracy** F1 和精确匹配（EM）是模型性能准确度的两个评估指标。F1 测量预测答案和 groundtruth 之间的重叠 tokens 部分，而如果预测与groundtruth 完全相同则精确匹配得分为 1，否则为 0。我们将结果与表2 中的其他方法进行比较。为了进行公平和彻底的比较，我们都会在最新的论文/预印本中报告已发布的结果，并在排行榜上报告更新但未记录的结果。我们认为后者是未发表的结果。从表中可以看出，我们模型的精度（EM / F1）性能与最先进的模型相当。特别地，我们在原始数据集上训练的模型在 EM 和 F1 得分方面优于文献中的所有记录结果（参见表2 的第二列）。当使用适当的采样方案训练增强数据时，我们的模型可以在 EM/F1 上获得显著的增益 1.5/1.1。最后，我们在官方测试集上的结果是 76.2/84.6，这明显优于最佳记录结果 73.2/81.8。  

![Aaron Swartz](https://github.com/liyibo/cv_notebooks/blob/master/markdown_pics/NLP/9/5.jpg?raw=true)

**Speedup over RNNs** 为了测量我们的模型相对于 RNN 模型的加速，我们还测试相应的模型架构，其中每个编码器块替换为大多数现有模型中使用的双向 LSTMs。具体地，每个（嵌入和模型）编码器块分别用 1,2 或 3 层双向 LSTM 替换，因为这样的层数落入阅读理解模型的通常范围（Chen等人，2017）。所有这些 LSTM 都有隐藏的大小 128。加速比较的结果如表3 所示。我们可以很容易地看到我们的模型明显快于所有基于 RNN 的模型，并且加速范围在训练中为 3 到 13 倍，测试中为 4 到 9 倍。  

![Aaron Swartz](https://github.com/liyibo/cv_notebooks/blob/master/markdown_pics/NLP/9/6.jpg?raw=true)

**Speedup over BiDAF model** 此外，我们还使用相同的硬件（NVIDIA p100 GPU），并比较我们的模型和 BiDAF 模型（Seo等，2016）之间获得相同性能的训练时间，这是 SQuAD 上经典的基于 RNN 的模型。我们大多采用原始代码中的默认设置来获得最佳性能，其中训练和推理的批量大小都是 60。我们改变的唯一部分是优化器，其中使用了学习 0.001 的 Adam。结果如表4 所示，表明我们的模型在训练和推理速度方面比 BiDAF 快 4.3 和 7.0 倍。此外，我们只需要五分之一的训练时间来获得 BiDAF 在 dev set 最佳 F1 得分（77.0）。  

#### 4.1.3 ABALATION STUDY AND ANALYSIS  

![Aaron Swartz](https://github.com/liyibo/cv_notebooks/blob/master/markdown_pics/NLP/9/7.jpg?raw=true)

我们对拟议模型的组成部分进行消融研究，并研究增强数据的影响。表5 中显示了 dev set 上的验证分数。从表中可以看出，在编码器中使用卷积是至关重要的：如果移除了卷积，F1 和 EM 将急剧下降近 3%。编码器中的自我关注也是一个必要的组件，它可以为最终性能提供 1.4/1.3 的 EM/F1 增益。我们将这些现象解释如下：卷积捕获上下文的局部结构，而自我关注能够模拟文本之间的全局交互。因此，他们互补但不能互相替换。使用可分离卷积代替传统卷积也对性能有显著贡献，这可以通过用正常卷积替换可分离卷积所引起的稍微差一些的精度来看出。  

**数据增强的影响** 我们另外进行实验以了解增强数据的数量增加时的值。正如表中最后一行的行显示的那样，数据增强证明有助于进一步提升性能。 通过仅添加 En-Fr-En 数据使训练数据大两倍（原始训练数据和增强数据之间的比率 1：1，如行“数据增加×2（1：1：0）”所示）得到 F1 增加 0.5%。 虽然使用法语作为枢轴添加更多增强数据并不能提供性能提升，但注入相同数量的额外增强数据 En-De-En 会使 F1 再次提高 0.2，如条目“数据增加×3（1：1：1）”。我们可以将这种收益归因于新数据的多样性，这是由新语言的翻译者产生的。  

**抽样方案的效果** 虽然注入超过 ×3 的更多数据对模型没有好处，但我们观察到在训练期间原始数据和增强数据之间的良好采样率可以进一步提高模型性能。特别是，当我们将增强数据的采样权重从（1：1：1）增加到（1：2：1）时，EM/F1性能下降 0.5/0.3。我们猜想这是因为增加的数据由于反向翻译而产生噪声，所以它不应该是训练的主要数据。我们通过将原始数据的比率从（1：2：1）增加到（2：2：1）来确认这一点，其中获得了对 EM/F1 的 0.6/0.5 性能增益。然后我们修复增强数据的一部分，并搜索原始数据的样本权重。根据经验，该比率（3：1：1）产生最佳性能，在 EM/F1 上比基本模型增加 1.5/1.1。这也是我们提交的用于测试集评估的模型。  

#### 4.1.4 ROBUSTNESS STUDY  

![Aaron Swartz](https://github.com/liyibo/cv_notebooks/blob/master/markdown_pics/NLP/9/8.jpg?raw=true)

在下文中，我们对对抗性 SQUAD 数据集（Jia＆Liang，2017）进行了实验，以研究所提出模型的稳健性。在该数据集中，将一个或多个句子附加到测试集的原始 SQuAD 上下文中，以故意误导训练的模型以产生错误的答案。但是，该模型在训练期间对那些对抗性示例是不可知的。我们关注两种类型的误导性句子，即 AddSent 和 AddOneSent。AddSent 生成与问题类似的句子，但与正确答案不矛盾，而 AddOneSent 添加了一个随机的人类提供的句子，该句子不一定与上下文相关。  

使用的模型正是使用原始 SQuAD 数据训练的模型（在测试集上得到84.6 F1），但现在它被提交给对抗服务器进行评估。结果如表6 所示，其他模型的 F1 得分均来自 Jia＆Liang（2017）。再次，我们只比较单个模型的表现。从表6 中我们可以看出，我们的模型与最先进的模型相当，而且大大优于其他模型。我们模型的稳健性可能是因为它是用增强数据训练的。训练数据中的注入噪声不仅可以改善模型的泛化，而且可以使其对抗对抗句。  

#### 4.2 EXPERIMENTS ON TRIVIAQA  

在本节中，我们在另一个数据集 TriviaQA（Joshi等，2017）上测试我们的模型，该数据集由 650K 上下文-查询-答案三元组组成。有 95K 个不同的问答配对，由 Trivia 爱好者撰写，平均每个问题有 6 个证据文件（上下文），可以从维基百科或网络搜索中抓取。与 SQUAD 相比，TriviaQA 更具挑战性：1）其示例具有更长的上下文（平均每个上下文 2895 个 tokens）并且可能包含多个段落，2）由于缺少人工标签，它比 SQuAD 噪声更大，3 ）上下文可能与答案无关，因为它被关键词抓取。  

在本文中，我们专注于在由维基百科的答案组成的子集上测试我们的模型。根据之前的工作（Joshi等人，2017; Hu等人，2017; Pan等人，2017），相同的模型在维基百科和网络上都有类似的表现，但后者是五倍大。为了使训练时间易于管理，我们省略了对 Web 数据的实验。  

由于上下文的多段性质，研究人员还发现简单的分层或多步读取技巧，例如首先预测要阅读的段落然后应用像 BiDAF 这样的模型来确定该段落中的答案（Clark＆Gardner， 2017），可以显著提升 TriviaQA 的性能。但是，在本文中，我们只关注与单段阅读基线的比较。我们相信我们的模型可以插入其他多段读取方法以实现类似或更好的性能，但它超出了本文的范围。  

Wikipedia 子数据集包含大约 92K 训练和 11K 开发示例。平均上下文和问题长度分别为 495 和 15。除了完整的开发集，Joshi 等人还挑选一个经过验证的子集，其中所有上下文都可以回答相关问题。由于文本可能很长，我们采用类似于 Hu 等人的数据处理。 特别是，对于训练和验证，我们随机选择一个长度为 256 和 400 的窗口，分别封装答案。除了训练步骤设置为 120K 之外，所有剩余设置与 SQuAD 实验相同。  

![Aaron Swartz](https://github.com/liyibo/cv_notebooks/blob/master/markdown_pics/NLP/9/9.jpg?raw=true)

**Accuracy** 表7 中显示了开发集的准确性。再次，我们可以看到我们的模型在完整开发集上的 F1 和 EM 方面优于基线，并且与最先进的模型相当。  

![Aaron Swartz](https://github.com/liyibo/cv_notebooks/blob/master/markdown_pics/NLP/9/10.jpg?raw=true)

**Speedup over RNNs** 除了准确性之外，我们还将模型的速度与 RNN 对应物的速度进行对比。如表8 所示，毫不奇怪，我们的模型在训练中具有 3 到 11 倍的加速，在推理中具有 3 到 9 倍的加速度，类似于 SQuAD 数据集中的发现。  

## 5 RELATED WORK  

机器阅读理解和自动问答已成为 NLP 领域的一个重要课题。它们的受欢迎程度可归因于公开可用的标注数据集的增加，例如 SQuAD（Rajpurkar等，2016），TriviaQA（Joshi等，2017），CNN/Daily News（Hermann等，2015），WikiReading（Hewlett等，2016），儿童书籍测试（Hill等，2015）等。已经提出了大量的端到端神经网络模型来应对这些挑战，包括 BiDAF（Seo et al，2016），r-net（Wang et al，2017），DCN（Xiong et al，2016），ReasoNet（Shen et al，2017b），Document Reader（Chen et al，2017），Interactive AoA Reader （Cui et al，2017）和 Reinforced Mnemonic Reader（Hu et al，2017）。  

在过去几年中，递归神经网络（RNN）在自然语言处理中占据主导地位。文本的顺序性质与 RNN 的设计理念一致，因此它们很受欢迎。事实上，上面提到的所有阅读理解模型都是基于 RNN 的。尽管是常见的，但 RNN 的顺序性质阻止了并行计算，因为必须按顺序将 tokens 馈送到 RNN 中。RNN 的另一个缺点是难以对长依赖性建模，尽管通过使用门控循环单元（Chung等，2014）或长短期记忆体系结构（Hochreiter＆Schmidhuber，1997）有所缓解。对于诸如文本分类之类的简单任务，使用强化学习技术，已经提出模型（Yu等人，2017）以跳过不相关的 tokens 以进一步解决长依赖性问题并加速该过程。但是，目前尚不清楚这些方法是否可以处理复杂的任务，如问答。本文考虑的阅读理解任务总是需要处理长文本，因为上下文段落可能长达数百个单词。最近，已经尝试通过全卷积或全注意力架构来取代循环网络（Kim，2014; Gehring等，2017; Vaswani等，2017b; Shen等，2017a）。已经证明这些模型不仅比 RNN 架构更快，而且在其他任务中也有效，例如文本分类，机器翻译或情感分析。  

据我们所知，本文是通过丢弃循环网络以支持前馈体系结构来实现快速准确的阅读理解模型的第一项工作。我们的论文也是第一个将自我关注和卷积相结合的论文，这证明了经验上的有效性并且实现了 2.7 F1 的显著增益。请注意，Raiman＆Miller（2017）最近提出通过避免双向注意并以 search beams 为条件来加速阅读理解。尽管如此，他们的模型仍然基于 RNN，并且准确性没有竞争力，EM 68.4 和 F1 76.2。Weissenborn 等还尝试通过删除上下文查询注意模块来构建快速 Q＆A 模型。然而，它再次依赖于 RNN，因此本质上比我们慢。消除注意力进一步牺牲了性能（EM 68.4 和 F1 77.1）。  

在自然语言处理中也已经探索了数据增加。例如，Zhang 等人提出通过用同义词替换单词来增强数据集，并在文本分类中显示其有效性。Raiman&Miller（2017）建议使用类型交换来扩充 SQuAD 数据集，它基本上将原始段落中的单词替换为具有相同类型的其他单词。虽然它被证明可以提高准确性，但增强数据具有与原始数据相同的句法结构，因此它们不够多样化。Zhou 等人通过产生更多问题改善了 SQUAD 数据的多样性。然而，正如 Wang 等人报道的那样，他们的方法对提升性能没有帮助。本文提出的数据增强技术是基于通过来回翻译原始文本来解释句子。主要的好处是它可以为增强的数据带来更多的语法多样性。  

## 6 CONCLUSION  

在本文中，我们提出了一种快速准确的端到端模型 QANet，用于机器阅读理解。我们的核心创新是完全删除编码器中的循环网络。得到的模型是完全前馈的，完全由可分离的卷积，注意力，线性层和层归一化组成，适用于并行计算。由此产生的模型既快速又准确：它超过了 SQuAD 数据集上发布的最佳结果，而比竞争性循环模型的训练/推理迭代快了 13/9 倍。此外，我们发现我们能够通过利用数据扩充来实现显著的收益，该数据扩充包括将上下文和通道对转换为来自另一种语言和来自另一种语言，作为解释问题和上下文的方式。

# Attention Is All You Need  

## Abstract  

主要序列转导模型基于包括编码器和解码器的复杂递归或卷积神经网络。性能最佳的模型还通过注意机制连接编码器和解码器。我们提出了一种新的简单网络架构，转换器完全基于注意机制，完全消除了循环和卷积。两个机器翻译任务的实验表明，这些模型在质量上更优越，同时更易于并行化，并且需要更少的时间进行训练。我们的模型在 WMT 2014 英语 - 德语翻译任务中达到 28.4 BLEU，超过现有的最佳成绩，包括 ensembles，超过 2 BLEU。在 WMT 2014 英语 - 法语翻译任务中，我们的模型在 8 个 GPU 上训练 3.5 天后，建立了一个新的单模型最新 BLEU 分数 41.8，这个训练成本只是文献中效果好的模型的一小部分。我们通过在大量和少量训练数据上将其成功应用于英语 constituency parsing，表明 Transformer 可以很好地推广到其他任务。  

## 1 Introduction  

循环神经网络，长短时记忆[13]和特别是门控循环[7]神经网络，已经成为用于序列建模和转换问题的最先进的方法，如语言建模和机器翻译。自那以后，许多努力继续推动循环语言模型和编码器-解码器架构的界限。  

递归模型通常考虑沿输入和输出序列的符号位置的计算。将位置与计算时间中的步骤对齐，它们产生一系列隐藏状态 ht，作为先前隐藏状态 ht-1 和位置 t 的输入的函数。这种固有的顺序特性排除了训练示例中的并行化，这在较长的序列长度中变得至关重要，因为内存约束限制了跨越示例的批处理。最近的工作通过分解技巧[21]和条件计算[32]实现了计算效率的显著提高，同时在后者的情况下也提高了模型性能。然而，顺序计算的基本约束仍然存在。  

注意机制已成为各种任务中引人注目的序列建模和转换模型的组成部分，允许对依赖关系进行建模，而不考虑它们在输入或输出序列中的距离[2,19]。然而，在少数情况下[27]，这种注意机制与循环网络结合使用。  

在这项工作中，我们提出了 Transformer，一种避免循环的模型架构，而是完全依赖于注意机制来绘制输入和输出之间的全局依赖关系。Transformer 允许更多的并行化，并且在 8 个 P100 GPU 上训练了长达 12 小时后，可以达到翻译质量的最新技术水平。  

## 2 Background  

减少顺序计算的目标也构成了 Extended Neural GPU [16]，ByteNet [18]和 ConvS2S [9]的基础，所有这些都使用卷积神经网络作为基本构建块，并行计算所有输入和输出的隐藏表示。在这些模型中，关联来自两个任意输入或输出位置的信号所需的操作数量在位置之间的距离上增长，对于 ConvS2S 呈线性增长，对于 ByteNet 呈线对数。这使得学习远程位置之间的依赖性变得更加困难[12]。在 Transformer 中，这被减少到恒定的操作次数，尽管由于平均注意加权位置而导致有效分辨率降低，这是我们在 3.2 节中描述的 Multi-Head Attention 的影响。  

Self-attention，有时称为内部注意力是关联单个序列的不同位置的注意机制，以便计算序列的表示。自我注意力已经成功地用于各种任务，包括阅读理解，抽象概括，文本蕴涵和学习任务独立的句子表示。  

End-to-end memory networks 基于循环注意机制而不是序列对齐，并且已被证明在简单语言问答和语言建模任务中表现良好[34]。  

然而，据我们所知，Transformer 是第一个完全依靠自我注意力的转换模型来计算其输入和输出的表示，而不使用序列对齐的 RNN 或卷积。在接下来的部分中，我们将描述 Transformer，motivate self-attention 并讨论其优于[17,18]和[9]等模型的优势。  

## 3 Model Architecture  

大多数较好的神经序列转换模型具有编码器-解码器结构[5,2,35]。这里，编码器将符号表示的输入序列 $(x_1,...,x_n)$ 映射到连续表示的序列 $z = (z_1,...,z_n)$。给定 z，解码器然后一次一个元素地生成符号的输出序列 $(y_1,...,y_m)$。在每个步骤中，模型都是自回归的[10]，在生成下一个时将先前生成的符号作为附加输入。Transformer 遵循这种整体架构，使用堆叠的 self-attention 和 point-wise，全连接层用于编码器和解码器，分别如图1 的左半部分和右半部分所示。  

### 3.1 Encoder and Decoder Stacks  

**编码器** 编码器由一堆 N = 6 个相同的层组成。每层有两个子层。第一层是多头自我注意力机制，第二种是简单的，位置全连接的前馈网络。我们在两个子层中的每一个周围使用残差连接[11]，然后是层归一化[1]。也就是说，每个子层的输出是 $LayerNorm(x + Sublayer(x))$，其中 $Sublayer(x)$ 是由子层本身实现的功能。为了促进这些残差连接，模型中的所有子层以及嵌入层产生维度 $d_{model} = 512$ 的输出。  

**解码器** 解码器也由 N = 6 个相同层的堆栈组成。除了每个编码器层中的两个子层之外，解码器还插入第三子层，其对编码器堆栈的输出执行多头注意。与编码器类似，我们在每个子层周围使用残差连接，然后进行层规范化。我们还修改解码器堆栈中的自注意子层以防止位置注意后续位置。这种掩蔽与输出嵌入偏移一个位置的事实相结合，确保了位置 i 的预测仅依赖于小于 i 的位置处的已知输出。  

### 3.2 Attention  

注意功能可以被描述为将查询和一组键值对映射到输出，其中查询，键，值和输出都是向量。 输出计算为值的加权和，其中分配给每个值的权重由查询与相应密钥的兼容性函数计算  

#### 3.2.1 Scaled Dot-Product Attention  

我们特别注意力“Scaled Dot-Product Attention”（图2）。 输入包括维度dk的查询和键以及维度dv的值。 我们用所有键计算查询的点积，将每个除以√dk，并应用softmax函数来获得值的权重。  

在实践中，我们同时在一组查询上计算注意函数，将它们打包在一起形成矩阵Q.键和值也一起打包成矩阵K和V. 我们计算输出矩阵为：  

两个最常用的注意功能是加性注意[2]和点积（乘法）注意。 除了缩放因子√1dk之外，点产品注意力与我们的算法相同。 附加注意使用具有单个隐藏层的前馈网络来计算兼容性功能。 虽然两者在理论上的复杂性相似，但在实践中，点积注意力更快，更节省空间，因为它可以使用高度优化的矩阵乘法码来实现。  

虽然对于较小的dk值，两种机制的表现相似，但附加注意力优于点积注意，而不会缩放较大的dk值[3]。 我们怀疑，对于较大的dk值，点积大幅增大，将softmax函数推向具有极小梯度的区域4。 为了抵消这种影响，我们将点积缩放√1dk。  

#### 3.2.2 Multi-Head Attention  

我们发现，使用dmodel-dimensional键，值和查询执行单个注意函数，我们发现将查询，键和值h次分别用不同的，学习的线性投影线性投影到dk，dk和dv维度是有益的。 然后，在这些投影版本的查询，键和值中，我们并行执行注意功能，从而产生dv维度输出值。 将它们连接起来并再次投影，得到最终值，如图2所示。  

多头注意允许模型共同注意力来自不同位置的不同表示子空间的信息。 只需一个注意头，平均就会抑制这种情况。  

其中投影是参数矩阵W Qi∈Rdmodel×dk，W Ki∈Rdmodel×dk，WVi∈Rdmodel×dv和WO∈Rhdv×dmodel。  

在这项工作中，我们采用h = 8个平行注意力层或头部。 对于这些中的每一个，我们使用dk = dv = dmodel / h = 64.由于每个头的尺寸减小，总计算成本与具有全维度的单头注意力相似。  

#### 3.2.3 Applications of Attention in our Model  

Transformer以三种不同的方式使用多头注意力：  

•在“编码器 - 解码器注意力”层中，查询来自先前的解码器层，存储器键和值来自编码器的输出。这允许解码器中的每个位置参与输入序列中的所有位置。这模仿了序列到序列模型中的典型编码器 - 解码器注意机制，例如[38,2,9]。
•编码器包含自我注意力层。在自我注意力层中的所有键，值
和查询来自同一个地方，在这种情况下，是编码器中前一层的输出。编码器中的每个位置都可以处理编码器前一层中的所有位置。
•类似地，解码器中的自注意层允许解码器中的每个位置处理解码器中的所有位置，直到并包括该位置。我们需要防止解码器中的向左信息流以保持自回归属性。我们通过屏蔽（设置为-∞）softmax输入中与非法连接相对应的所有值来实现缩放点产品注意内部。见图2。  

### 3.3 Position-wise Feed-Forward Networks  

除了注意力子层之外，我们的编码器和解码器中的每个层都包含一个完全连接的前馈网络，该网络分别和相同地应用于每个位置。 这包括两个线性变换，其间有ReLU激活。  

虽然线性变换在不同位置上是相同的，但它们在层与层之间使用不同的参数。 另一种描述这种情况的方法是两个内核大小为1的卷积。输入和输出的维数是dmodel = 512，内层的维度df f = 2048。  

### 3.4 Embeddings and Softmax  

与其他序列转导模型类似，我们使用学习嵌入将输入标记和输出标记转换为维度dmodel的向量。 我们还使用通常学习的线性变换和softmax函数将解码器输出转换为预测的下一个令牌概率。 在我们的模型中，我们在两个嵌入层和pre-softmax线性变换之间共享相同的权重矩阵，类似于[30]。 在嵌入层中，我们将这些权重乘以√d模型。  

### 3.5 Positional Encoding  

由于我们的模型不包含递归和卷积，为了使模型能够利用序列的顺序，我们必须注入关于序列中令牌的相对或绝对位置的一些信息。 为此，我们将“位置编码”添加到编码器和解码器堆栈底部的输入嵌入。 位置编码具有与嵌入相同的维度dmodel，因此可以对两者进行求和。 有许多位置编码选择，学习和固定[9]。  

在这项工作中，我们使用不同频率的正弦和余弦函数：  

pos是位置，我是维度。 也就是说，位置编码的每个维度对应于正弦曲线。 波长形成从2π到10000·2π的几何级数。 我们之所以选择这个函数，是因为我们假设它可以让模型轻松地学习相对位置，因为对于任何固定偏移k，P Epos + k可以表示为P Epos的线性函数。  

我们还尝试使用学习的位置嵌入[9]，并发现这两个版本产生了几乎相同的结果（参见表3第（E）行）。 我们选择了正弦曲线版本，因为它可以允许模型外推到比训练期间遇到的序列长度更长的序列长度。  

## 4 Why Self-Attention  

在本节中，我们将自我注意力层的各个方面与通常用于将一个可变长度符号表示序列（x1，...，xn）映射到另一个相等长度序列（z1，...）的循环和卷积层进行比较。 。，zn），其中xi，zi∈Rd，例如典型序列转导编码器或解码器中的隐藏层。 激励我们使用自我注意力，我们考虑三个需求。  

一个是每层的总计算复杂度。 另一个是可以并行化的计算量，通过所需的最小顺序操作数来衡量。  

第三个是网络中远程依赖之间的路径长度。 学习远程依赖性是许多序列转导任务中的关键挑战。 影响学习这种依赖性的能力的一个关键因素是前向和后向信号必须在网络中传播的路径的长度。 输入和输出序列中任何位置组合之间的这些路径越短，学习远程依赖性就越容易[12]。 因此，我们还比较了由不同层类型组成的网络中任意两个输入和输出位置之间的最大路径长度。  

如表1所示，自我注意力层通过恒定数量的顺序执行操作连接所有位置，而复发层需要O（n）个顺序操作。 就计算复杂性而言，当序列长度n小于表示维度d时，自注意层比复发层更快，这通常是机器翻译中最先进模型使用的句子表示的情况。 ，例如字组[38]和字节对[31]表示。 为了提高涉及非常长序列的任务的计算性能，可以将自我注意力限制为仅考虑以相应输出位置为中心的输入序列中的大小为r的邻域。 这会将最大路径长度增加到O（n / r）。 我们计划在未来的工作中进一步研究这种方法。  

内核宽度为k n的单个卷积层不会连接所有输入和输出位置对。 这样做需要在连续内核的情况下堆叠O（n / k）卷积层，或者在扩张卷积的情况下需要O（logk（n））[18]，增加任意两个位置之间的最长路径的长度 在网络中。 卷积层通常比复发层更昂贵，为k倍。 然而，可分离的卷积[6]大大降低了复杂度O（k·n·d + n·d 2）。 然而，即使k = n，可分离卷积的复杂性也等于自注意层和逐点前馈层的组合，这是我们在模型中采用的方法。  

作为附带利益，自我注意力可以产生更多可解释的模型。 我们检查模型中的注意力分布，并在附录中展示和讨论示例。 不仅个别注意头明确地学会执行不同的任务，许多人似乎表现出与句子的句法和语义结构相关的行为。  

## 5 Training  

本节介绍了我们模型的培训制度。  

### 5.1 Training Data and Batching  

我们使用标准WMT 2014英语 - 德语数据集进行了培训，该数据集包含大约450万个句子对。 句子使用字节对编码[3]编码，其具有大约37000个令牌的共享源目标词汇表。 对于英语 - 法语，我们使用了大得多的WMT 2014英语 - 法语数据集，该数据集由36M个句子组成，并将令牌分成32000个单词词汇[38]。 句子对按照近似的序列长度进行批处理。 每个训练批包含一组句子对，包含大约25000个源代币和25000个目标代币。  

### 5.2 Hardware and Schedule  

我们在一台配备8个NVIDIA P100 GPU的机器上训练我们的模型。 对于使用本文所述的超参数的基本模型，每个训练步骤大约需要0.4秒。 我们对基础模型进行了总共100,000步或12小时的培训。 对于我们的大型机型（在表3的底线描述），步进时间为1.0秒。 大型模型经过300,000步（3.5天）的培训。  

### 5.3 Optimizer  

我们使用Adam优化器[20]，β1= 0.9，β2= 0.98和？ = 10-9。 根据以下公式，我们在培训过程中改变了学习率：  

这对应于为第一个warmup_steps训练步骤线性地增加学习速率，然后与步数的反平方根成比例地减小它。 我们使用了warmup_steps = 4000。  

### 5.4 Regularization  

我们在培训期间采用三种正规化：  

剩余丢失我们将dropout [33]应用于每个子层的输出，然后将其添加到子层输入并进行标准化。 此外，我们将丢包应用于编码器和解码器堆栈中的嵌入和位置编码的总和。 对于基本模型，我们使用Pdrop = 0.1的速率。  

标签平滑在训练过程中，我们采用了标签平滑值？ls = 0.1 [36]。 这会伤害困惑，因为模型学会更加不确定，但提高了准确性和BLEU分数。  

## 6 Results  

### 6.1 Machine Translation  

在WMT 2014英语 - 德语翻译任务中，大变压器模型（表2中的变压器（大））优于先前报告的最佳模型（包括合奏）超过2.0 BLEU，建立了一个新的状态 - 艺术BLEU得分为28.4。 该模型的配置列于表3的底部。在8个P100 GPU上，培训需要3.5天。 甚至我们的基础模型也超过了之前发布的所有模型和合奏，而且只占培训成本的一小部分。  

在WMT 2014英语 - 法语翻译任务中，我们的大型模型获得了41.0的BLEU分数，优于以前发布的所有单一模型，不到以前最先进技术培训成本的1/4 模型。 使用英语到法语训练的变形金刚（大）模型使用辍学率Pdrop = 0.1，而不是0.3。  

对于基本模型，我们使用通过平均最后5个检查点获得的单个模型，这些检查点以10分钟的间隔写入。 对于大型模型，我们平均了最后20个检查点。 我们使用光束搜索，光束大小为4，长度罚分α= 0.6 [38]。 在开发集上进行实验后选择这些超参数。 我们将推理期间的最大输出长度设置为输入长度+ 50，但在可能的情况下尽早终止[38]。  

表2总结了我们的结果，并将我们的翻译质量和培训成本与文献中的其他模型架构进行了比较。 我们通过将训练时间，所使用的GPU的数量和每个GPU 5的持续单精度浮点容量的估计相乘来估计用于训练模型的浮点运算的数量。  

### 6.2 Model Variations  

为了评估变压器的不同组件的重要性，我们以不同的方式改变我们的基本模型，测量开发集上的英语 - 德语翻译的性能变化，newstest2013。 我们使用了上一节中描述的波束搜索，但没有检查点平均值。 我们将这些结果呈现在表3中。  

在表3的行（A）中，我们改变注意头的数量以及注意键和值的维度，保持计算量不变，如第3.2.2节所述。 虽然单头注意力比最佳设置低了0.9 BLEU，但是头部太多也会降低质量。  

在表3行（B）中，我们观察到减小注意力大小dk会损害模型质量。 这表明确定兼容性并不容易，并且比点积更复杂的兼容性功能可能是有益的。 我们在行（C）和（D）中进一步观察到，正如预期的那样，更大的模型更好，并且辍学对于避免过度拟合非常有帮助。 在行（E）中，我们用学习的位置嵌入[9]替换我们的正弦位置编码，并观察到与基本模型几乎相同的结果。  

### 6.3 English Constituency Parsing  

为了评估变形金刚是否可以推广到其他任务，我们对英语选区分析进行了实验。 这项任务提出了具体的挑战：产出受到强烈的结构限制，并且明显长于输入。 此外，RNN序列到序列模型还无法在小数据体系中获得最先进的结果[37]。  

我们在Penn Treebank [25]的华尔街日报（WSJ）部分训练了一个dmodel = 1024的4层变压器，大约40K训练句。 我们还使用较大的高可信度和BerkleyParser语料库以及大约17M的句子在半监督环境中训练它[37]。 我们使用16K令牌的词汇表仅用于WSJ设置，并使用32K令牌的词汇表用于半监督设置。  

我们仅进行了少量实验，以选择第22节开发集中的注意力和残差（第5.4节），学习率和光束大小，所有其他参数在英语 - 德语基础翻译模型中保持不变。 在推理过程中，我们将最大输出长度增加到输入长度+ 300.我们使用的光束尺寸为21，α= 0.3仅适用于WSJ和半监督设置。  

我们在表4中的结果表明，尽管缺乏任务特定的调整，我们的模型表现出色得多，产生的结果比之前报道的所有模型都要好，但回归神经网络语法除外[8]。  

与RNN序列到序列模型[37]相比，Transformer的表现优于BerkeleyParser [29]，即使只训练40K句子的WSJ训练集也是如此。  

## 7 Conclusion  

在这项工作中，我们提出了变压器，这是第一个完全基于注意力的序列转换模型，用多头自我注意力取代了编码器 - 解码器架构中最常用的复现层。  

对于转换任务，Transformer的训练速度明显快于基于循环或卷积层的架构。 在WMT 2014英语 - 德语和WMT 2014英语 - 法语翻译任务中，我们实现了最新的技术水平。 在之前的任务中，我们的最佳模型甚至优于以前报告过的所有合奏。  

我们对基于注意力的模型的未来感到兴奋，并计划将它们应用于其他任务。 我们计划将变换器扩展到涉及文本以外的输入和输出模态的问题，并研究局部的，受限制的注意机制，以有效地处理大型输入和输出，如图像，音频和视频。 让生成顺序更少是我们的另一个研究目标  

我们用于训练和评估模型的代码可在 https://github.com/tensorflow/tensor2tensor 上找到。

## 参考  

- 1 [Visualizing and Understanding Neural Models in NLP](https://arxiv.org/abs/1506.01066)  
- 2 [Distributed Representations of Words and Phrases and their Compositionality](https://arxiv.org/abs/1310.4546)  
- 3 [Word2Vec Tutorial](http://mccormickml.com/2016/04/19/word2vec-tutorial-the-skip-gram-model/)  
- 4 [Enriching Word Vectors with Subword Information](https://arxiv.org/abs/1607.04606)  
- 5 [Bag of Tricks for Efficient Text Classification](https://arxiv.org/abs/1607.01759)  
- 6 [Massive Exploration of Neural Machine Translation Architectures](https://arxiv.org/abs/1703.03906)  
- 6 [A Neural Attention Model for Abstractive Sentence Summarization](https://arxiv.org/abs/1509.00685)  
- 7 [Abstractive Text Summarization Using Sequence-to-Sequence RNNs and Beyond](https://arxiv.org/abs/1602.06023)  
- 8 [QANet: Combining Local Convolution with Global Self-Attention for Reading Comprehension](https://arxiv.org/abs/1804.09541)  
- 9 [Attention Is All You Need](https://arxiv.org/abs/1706.03762)  